# Residual-Based Feature Boosting PoC

VS Code/Jupyter에서 위에서 아래로 실행하는 노트북입니다.

핵심 흐름:

1. `Xb` base feature로 baseline 수율 회귀 모델 학습
2. `baseline_residual = y - baseline_pred` 계산
3. defect별 bad/good group 기준으로 candidate feature 품질 필터링
4. candidate feature 하나만 사용해 current residual 예측
5. validation bad group의 설정 metric reduction 기준으로 feature 선택
6. 선택 feature를 `Xb + selected Xnew`에 추가해 final CatBoost 재학습
7. baseline vs final metric, ranking, SHAP summary 저장

중요: residual boosting 단계에서는 `Xb`를 다시 학습하지 않습니다. 후보 feature `x_j` 하나만 residual model에 사용합니다.

## 0. 환경 준비

**이 셀에서 하는 일**

- 현재 노트북이 어떤 폴더에서 실행되든 repo root를 찾아 작업 경로로 이동합니다.
- `feature_boosting` 패키지를 import할 수 있도록 `sys.path`를 설정합니다.
- 이후 셀에서 사용할 함수들을 미리 import합니다.

**입력**: 없음  
**출력**: `ROOT` 경로 출력



In [ ]:
from pathlib import Path
import os
import sys
import time

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "feature_boosting").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("repo root not found")
    ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from feature_boosting.answer_features import add_answer_feature_flags, answer_feature_mask
from feature_boosting.baseline_model import add_baseline_predictions, metrics_by_split, train_baseline_model
from feature_boosting.data_loader import align_base_and_candidates, candidate_feature_cols, load_base_dataset, load_base_feature_cols, load_candidate_features, load_group_ids, standardize_six_file_inputs
from feature_boosting.final_model import evaluate_model_by_groups, predict_final, train_final_model
from feature_boosting.metrics import mae, r2, rmse
from feature_boosting.reporting import baseline_residual_summary, final_metric_summary, iteration_residual_summary, plot_candidate_loss_ranking, plot_final_feature_set_summary, plot_final_metric_comparison, plot_residual_curve, plot_round_residual_points, prepare_output_dir, round_residual_summary, write_csv
from feature_boosting.residual_boosting import ResidualFeatureBooster, ResidualFeatureBoosterConfig
from feature_boosting.overfit import recommend_overfit_safe_settings
from feature_boosting.shap_analysis import compute_shap_summary
from feature_boosting.splitter import split_frame
from feature_boosting.validation import profile_candidate_features, validate_defect_groups, validate_input_columns
from feature_boosting.config import FeatureFilterConfig

print("ROOT =", ROOT)


## 1. 실험 설정

**이 셀에서 하는 일**

- toyset을 쓸지 실제 데이터를 쓸지 선택합니다.
- toyset 크기(`DEMO_N_WAFERS`, `DEMO_N_CANDIDATE_FEATURES`)를 정합니다.
- 실제 데이터 경로, defect bad/good group 경로, output 경로를 정의합니다.
- baseline/residual/final 모델 파라미터와 feature filter 기준을 정합니다.
- candidate scoring 진행률을 표시할지와 출력 간격을 정합니다.

기본값은 `USE_DEMO_DATA = True`입니다. 그래서 repo를 clone한 직후 실제 데이터가 없어도 toyset을 자동 생성해서 바로 실행됩니다.

실제 데이터가 준비되어 있으면 `USE_DEMO_DATA = False`로 바꾸고 아래 경로만 수정하면 됩니다.

실제 데이터가 6개 raw 파일 구조(`lot/wf/y`, candidate, base feature, defect별 `good_bad`)라면 `USE_RAW_SIX_FILE_DATA = True`로 바꾸고 `RAW_*` 경로와 컬럼명을 맞추면 됩니다.

CatBoost가 설치되어 있으면 `backend: "auto"`에서 CatBoost를 사용합니다. CatBoost만 강제하려면 `backend: "catboost"`로 바꾸세요.

**입력**: 사용자가 수정하는 설정값  
**출력**: `OUT_DIR` 생성 및 경로 출력



In [ ]:
USE_DEMO_DATA = True
USE_RAW_SIX_FILE_DATA = False  # Set True when using the real six-file raw input format.
DEMO_N_WAFERS = 2_000
DEMO_N_CANDIDATE_FEATURES = 100  # total candidate feature count, including hidden_defect_1/2
DEMO_RANDOM_SEED = 42
RUN_ID = "feature_boosting_rev0_notebook"

ID_COL = "sample_id"
TARGET_COL = "yield"
SPLIT_COL = "split"

# Default paths for already-standardized inputs.
BASE_DATASET_PATH = ROOT / "data" / "base_dataset.csv"
CANDIDATE_FEATURES_PATH = ROOT / "data" / "candidate_features.csv"
BASE_FEATURE_COLS_PATH = ROOT / "data" / "base_feature_cols.txt"
OUTPUT_BASE_DIR = ROOT / "outputs"

DEFECTS = [
    {
        "defect_id": "defect_1",
        "bad_group_path": ROOT / "data" / "groups" / "defect_1_bad.csv",
        "good_group_path": ROOT / "data" / "groups" / "defect_1_good.csv",
    },
    {
        "defect_id": "defect_2",
        "bad_group_path": ROOT / "data" / "groups" / "defect_2_bad.csv",
        "good_group_path": ROOT / "data" / "groups" / "defect_2_good.csv",
    },
    {
        "defect_id": "defect_3",
        "bad_group_path": ROOT / "data" / "groups" / "defect_3_bad.csv",
        "good_group_path": ROOT / "data" / "groups" / "defect_3_good.csv",
    },
]

# Six-file raw input mode. Edit each file block independently.
# Key rule per file:
# - If lot_id and wf_id are separate columns: set combined_id_col=None, id_col=None.
# - If one column such as lot_wf_id exists: set combined_id_col="lot_wf_id".
# - Use id_col only when the file already has a final sample_id-like key that should not be split.
RAW_STANDARDIZED_DIR = ROOT / "data" / "standardized_from_raw"
RAW_COMBINED_ID_SEP = "_"

RAW_Y_FILE = {
    "path": ROOT / "data" / "raw" / "y.csv",
    "id_col": None,
    "combined_id_col": None,
    "lot_col": "lot_id",
    "wf_col": "wf_id",
    "target_col": "y",
    "split_col": None,  # Set to train/valid/test column name if y.csv already has one.
}

RAW_CANDIDATE_FILE = {
    "path": ROOT / "data" / "raw" / "candidate_features.csv",
    "id_col": None,
    "combined_id_col": None,
    "lot_col": "lot_id",
    "wf_col": "wf_id",
}

RAW_BASE_FEATURE_FILE = {
    "path": ROOT / "data" / "raw" / "base_features.csv",
    "id_col": None,
    "combined_id_col": None,
    "lot_col": "lot_id",
    "wf_col": "wf_id",
}

RAW_SPLIT = {
    "train_ratio": 0.6,
    "valid_ratio": 0.2,
    "seed": 42,
}

RAW_DEFECT_GROUPS = [
    {
        "defect_id": "defect_1",
        "group_path": ROOT / "data" / "raw" / "defect_1_good_bad.csv",
        "id_col": None,
        "combined_id_col": None,
        "lot_col": "lot_id",
        "wf_col": "wf_id",
        "label_col": "good_bad",
        "good_value": "good",
        "bad_value": "bad",
    },
    {
        "defect_id": "defect_2",
        "group_path": ROOT / "data" / "raw" / "defect_2_good_bad.csv",
        "id_col": None,
        "combined_id_col": None,
        "lot_col": "lot_id",
        "wf_col": "wf_id",
        "label_col": "good_bad",
        "good_value": "good",
        "bad_value": "bad",
    },
    {
        "defect_id": "defect_3",
        "group_path": ROOT / "data" / "raw" / "defect_3_good_bad.csv",
        "id_col": None,
        "combined_id_col": None,
        "lot_col": "lot_id",
        "wf_col": "wf_id",
        "label_col": "good_bad",
        "good_value": "good",
        "bad_value": "bad",
    },
]

EXPERIMENT_METRIC = "MAE"  # "MAE" or "RMSE"
EXPERIMENT_METRIC_LOWER = EXPERIMENT_METRIC.strip().lower()
if EXPERIMENT_METRIC_LOWER not in {"mae", "rmse"}:
    raise ValueError("EXPERIMENT_METRIC must be 'MAE' or 'RMSE'")

BASELINE_MODEL_PARAMS = {
    "backend": "auto",
    "iterations": 3000,
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": EXPERIMENT_METRIC,
    "eval_metric": EXPERIMENT_METRIC,
    "random_seed": 42,
    "early_stopping_rounds": 100,
    "verbose": 200,
}

RESIDUAL_MODEL_PARAMS = {
    "backend": "auto",
    "iterations": 300,
    "depth": 3,
    "learning_rate": 0.05,
    "loss_function": EXPERIMENT_METRIC,
    "eval_metric": EXPERIMENT_METRIC,
    "random_seed": 42,
    "early_stopping_rounds": 30,
    "verbose": False,
    "thread_count": 1,
}

FINAL_MODEL_PARAMS = dict(BASELINE_MODEL_PARAMS)

FEATURE_FILTER = FeatureFilterConfig(
    max_missing_rate=0.8,
    min_unique_values=2,
)

BOOSTING_N_ROUNDS = 5

# Feature selection knobs.
# Rank mode example: BOOSTING_SELECTION_MODE="top_k", SELECT_PER_ROUND=20
# Ratio threshold example: BOOSTING_SELECTION_MODE="threshold", BOOSTING_SELECTION_METRIC=f"bad_{EXPERIMENT_METRIC_LOWER}_after_over_baseline", BOOSTING_SELECTION_THRESHOLD=0.3
SELECT_PER_ROUND = 1
BOOSTING_SELECTION_MODE = "top_k"  # "top_k" or "threshold"
BOOSTING_SELECTION_METRIC = f"bad_{EXPERIMENT_METRIC_LOWER}_reduction"
BOOSTING_SELECTION_THRESHOLD = None
BOOSTING_SELECTION_DIRECTION = "auto"
BOOSTING_MAX_SELECT_PER_ROUND = None
MIN_IMPROVEMENT = 0.0
MIN_VALID_BAD_SAMPLES = 1
BOOSTING_SHOW_PROGRESS = True
BOOSTING_PROGRESS_EVERY = 100

# Overfit guard: reject candidates that improve train residual but hurt validation.
AUTO_OVERFIT_SAFE_SETTINGS = True
OVERFIT_GUARD_ENABLED = True
OVERFIT_GUARD_METRIC_SCOPE = "bad"  # "bad", "good", or "global"
OVERFIT_GUARD_MIN_VALID_REDUCTION = 0.0
OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE = 1.0
OVERFIT_GUARD_MAX_VALID_TRAIN_GAP = 0.25
OVERFIT_GUARD_USE_TEST = False  # exploratory only; True uses test as an additional guard
OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE = 1.05

# Candidate rank chart y-axis: after residual / base-feature-only baseline residual.
CANDIDATE_LOSS_GLOBAL_METRIC_COL = f"valid_global_{EXPERIMENT_METRIC_LOWER}_after_over_baseline"
CANDIDATE_LOSS_BAD_METRIC_COL = f"valid_bad_{EXPERIMENT_METRIC_LOWER}_after_over_baseline"

# Optional known answer features per defect.
# You can mix exact names and flexible matching rules.
# Examples:
# - "exact_feature_name"
# - {"contains_all": ["abc", "step2"]}  # feature name contains both abc and step2
# - {"contains_any": ["abc", "defect"]}
# - {"regex": r"abc.*step2"}
ANSWER_FEATURES = {
    "defect_1": ["hidden_defect_1", {"contains_all": ["hidden", "1"]}],
    "defect_2": ["hidden_defect_2"],
    "defect_3": [],
}

SHAP_ENABLED = True
SHAP_MAX_SAMPLES = 5000

OUT_DIR = prepare_output_dir(OUTPUT_BASE_DIR, RUN_ID)
print("OUT_DIR =", OUT_DIR)


## 2. 선택 사항: 데모 데이터 생성

**이 셀에서 하는 일**

- `USE_DEMO_DATA = True`이면 synthetic toyset을 생성합니다.
- `USE_RAW_SIX_FILE_DATA = True`이면 실제 6개 raw 파일을 표준 입력 파일로 변환합니다.
- wafer/sample row, baseline feature, candidate feature, target `yield`를 만듭니다.
- `hidden_defect_1`, `hidden_defect_2`는 residual에서 잡혀야 하는 planted signal입니다.
- defect별 bad/good sample list CSV를 생성하고, 이후 셀이 그 파일을 읽도록 경로를 바꿉니다.

`USE_DEMO_DATA = True`이면 아래 셀이 toyset CSV를 자동으로 생성합니다. clone 직후에는 이 기본값 그대로 실행하면 됩니다.

**입력**: `DEMO_N_WAFERS`, `DEMO_N_CANDIDATE_FEATURES`, `DEMO_RANDOM_SEED` 또는 `RAW_*` 경로/컬럼명  
**출력**: `data/residual_poc_demo/` 또는 `data/standardized_from_raw/` 아래 CSV 파일들



In [ ]:
if USE_DEMO_DATA and USE_RAW_SIX_FILE_DATA:
    raise ValueError("USE_DEMO_DATA and USE_RAW_SIX_FILE_DATA cannot both be True")

if USE_DEMO_DATA:
    demo_dir = ROOT / "data" / "residual_poc_demo"
    group_dir = demo_dir / "groups"
    group_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(DEMO_RANDOM_SEED)
    n = DEMO_N_WAFERS
    n_noise_features = max(0, DEMO_N_CANDIDATE_FEATURES - 2)
    sample_id = np.array([f"WF_{idx:05d}" for idx in range(n)])
    n_train = int(round(n * 0.6))
    n_valid = int(round(n * 0.2))
    n_test = n - n_train - n_valid
    split = np.array(["train"] * n_train + ["valid"] * n_valid + ["test"] * n_test)
    rng.shuffle(split)

    base_temp = rng.normal(0, 1, n)
    base_pressure = rng.normal(0, 1, n)
    hidden_defect_1 = rng.normal(0, 1, n)
    hidden_defect_2 = rng.normal(0, 1, n)
    noise_candidates = {f"cand_noise_{i:04d}": rng.normal(0, 1, n) for i in range(n_noise_features)}

    y = 80 + 5 * base_temp - 3 * base_pressure + 6 * hidden_defect_1 - 4 * hidden_defect_2 + rng.normal(0, 0.3, n)

    base_df = pd.DataFrame(
        {
            ID_COL: sample_id,
            TARGET_COL: y,
            SPLIT_COL: split,
            "base_temp": base_temp,
            "base_pressure": base_pressure,
        }
    )
    candidate_df = pd.DataFrame(
        {
            ID_COL: sample_id,
            "hidden_defect_1": hidden_defect_1,
            "hidden_defect_2": hidden_defect_2,
            **noise_candidates,
        }
    )

    BASE_DATASET_PATH = demo_dir / "base_dataset.csv"
    CANDIDATE_FEATURES_PATH = demo_dir / "candidate_features.csv"
    BASE_FEATURE_COLS_PATH = demo_dir / "base_feature_cols.txt"
    base_df.to_csv(BASE_DATASET_PATH, index=False, encoding="utf-8-sig")
    candidate_df.to_csv(CANDIDATE_FEATURES_PATH, index=False, encoding="utf-8-sig")
    BASE_FEATURE_COLS_PATH.write_text("base_temp\nbase_pressure\n", encoding="utf-8")

    bad1 = base_df.loc[hidden_defect_1 >= np.quantile(hidden_defect_1, 0.75), [ID_COL]]
    good1 = base_df.loc[hidden_defect_1 < np.quantile(hidden_defect_1, 0.50), [ID_COL]]
    bad2 = base_df.loc[hidden_defect_2 <= np.quantile(hidden_defect_2, 0.25), [ID_COL]]
    good2 = base_df.loc[hidden_defect_2 > np.quantile(hidden_defect_2, 0.50), [ID_COL]]
    bad3 = base_df.sample(min(max(1, n // 5), len(base_df)), random_state=DEMO_RANDOM_SEED)[[ID_COL]]
    good3_pool = base_df.drop(bad3.index)
    good3 = good3_pool.sample(min(max(1, n // 3), len(good3_pool)), random_state=DEMO_RANDOM_SEED + 1)[[ID_COL]]

    bad1.to_csv(group_dir / "defect_1_bad.csv", index=False, encoding="utf-8-sig")
    good1.to_csv(group_dir / "defect_1_good.csv", index=False, encoding="utf-8-sig")
    bad2.to_csv(group_dir / "defect_2_bad.csv", index=False, encoding="utf-8-sig")
    good2.to_csv(group_dir / "defect_2_good.csv", index=False, encoding="utf-8-sig")
    bad3.to_csv(group_dir / "defect_3_bad.csv", index=False, encoding="utf-8-sig")
    good3.to_csv(group_dir / "defect_3_good.csv", index=False, encoding="utf-8-sig")

    DEFECTS = [
        {"defect_id": "defect_1", "bad_group_path": group_dir / "defect_1_bad.csv", "good_group_path": group_dir / "defect_1_good.csv"},
        {"defect_id": "defect_2", "bad_group_path": group_dir / "defect_2_bad.csv", "good_group_path": group_dir / "defect_2_good.csv"},
        {"defect_id": "defect_3", "bad_group_path": group_dir / "defect_3_bad.csv", "good_group_path": group_dir / "defect_3_good.csv"},
    ]
    print("demo data written to", demo_dir)
    print("demo wafers:", len(base_df), "| demo candidate features:", candidate_df.shape[1] - 1)

elif USE_RAW_SIX_FILE_DATA:
    standardized = standardize_six_file_inputs(
        y_path=RAW_Y_FILE["path"],
        candidate_path=RAW_CANDIDATE_FILE["path"],
        base_feature_path=RAW_BASE_FEATURE_FILE["path"],
        defect_groups=RAW_DEFECT_GROUPS,
        output_dir=RAW_STANDARDIZED_DIR,
        id_col=ID_COL,
        target_col=TARGET_COL,
        split_col=SPLIT_COL,
        y_lot_col=RAW_Y_FILE["lot_col"],
        y_wf_col=RAW_Y_FILE["wf_col"],
        y_id_col=RAW_Y_FILE["id_col"],
        y_combined_id_col=RAW_Y_FILE["combined_id_col"],
        y_target_col=RAW_Y_FILE["target_col"],
        candidate_lot_col=RAW_CANDIDATE_FILE["lot_col"],
        candidate_wf_col=RAW_CANDIDATE_FILE["wf_col"],
        candidate_id_col=RAW_CANDIDATE_FILE["id_col"],
        candidate_combined_id_col=RAW_CANDIDATE_FILE["combined_id_col"],
        base_lot_col=RAW_BASE_FEATURE_FILE["lot_col"],
        base_wf_col=RAW_BASE_FEATURE_FILE["wf_col"],
        base_id_col=RAW_BASE_FEATURE_FILE["id_col"],
        base_combined_id_col=RAW_BASE_FEATURE_FILE["combined_id_col"],
        combined_id_sep=RAW_COMBINED_ID_SEP,
        split_source_col=RAW_Y_FILE["split_col"],
        train_ratio=RAW_SPLIT["train_ratio"],
        valid_ratio=RAW_SPLIT["valid_ratio"],
        split_seed=RAW_SPLIT["seed"],
    )
    BASE_DATASET_PATH = standardized["base_dataset"]
    CANDIDATE_FEATURES_PATH = standardized["candidate_features"]
    BASE_FEATURE_COLS_PATH = standardized["base_feature_cols"]
    DEFECTS = standardized["defects"]
    print("raw 6-file data standardized to", RAW_STANDARDIZED_DIR)
    print("base dataset:", BASE_DATASET_PATH)
    print("candidate features:", CANDIDATE_FEATURES_PATH)
    print("defects:", [item["defect_id"] for item in DEFECTS])

else:
    print("using existing standardized input paths")
    print("base dataset:", BASE_DATASET_PATH)
    print("candidate features:", CANDIDATE_FEATURES_PATH)
    print("base feature cols:", BASE_FEATURE_COLS_PATH)


## 3. 데이터 로드 및 검증

**이 셀에서 하는 일**

- base dataset, candidate feature dataset, base feature column list를 읽습니다.
- 필수 컬럼(`sample_id`, `yield`, `split`)이 있는지 확인합니다.
- base dataset과 candidate feature를 `sample_id` 기준으로 join합니다.
- `split` 컬럼을 기준으로 train/valid/test DataFrame을 나눕니다.

**입력**: base/candidate/base_feature_cols 파일  
**출력**: `all_df`, `train_df`, `valid_df`, `test_df`, `base_feature_cols`, `candidate_cols`



In [ ]:
base_df = load_base_dataset(BASE_DATASET_PATH)
candidate_df = load_candidate_features(CANDIDATE_FEATURES_PATH)
base_feature_cols = load_base_feature_cols(BASE_FEATURE_COLS_PATH)
candidate_cols = candidate_feature_cols(candidate_df, ID_COL)

validate_input_columns(
    base_df,
    candidate_df,
    base_feature_cols,
    id_col=ID_COL,
    target_col=TARGET_COL,
    split_col=SPLIT_COL,
)

all_df = align_base_and_candidates(base_df, candidate_df, ID_COL)
train_df, valid_df, test_df = split_frame(all_df, SPLIT_COL)

print("base_df:", base_df.shape)
print("candidate_df:", candidate_df.shape)
print("merged:", all_df.shape)
print("base features:", len(base_feature_cols))
print("candidate features:", len(candidate_cols))
print(all_df[SPLIT_COL].value_counts())

if AUTO_OVERFIT_SAFE_SETTINGS:
    overfit_recommendation = recommend_overfit_safe_settings(
        RESIDUAL_MODEL_PARAMS,
        n_train_rows=len(train_df),
        n_candidate_features=len(candidate_cols),
        n_base_features=len(base_feature_cols),
        metric_name=EXPERIMENT_METRIC_LOWER,
    )
    RESIDUAL_MODEL_PARAMS = overfit_recommendation["residual_model_params"]
    guard = overfit_recommendation["guard"]
    OVERFIT_GUARD_ENABLED = guard["overfit_guard_enabled"]
    OVERFIT_GUARD_METRIC_SCOPE = guard["overfit_guard_metric_scope"]
    OVERFIT_GUARD_MIN_VALID_REDUCTION = guard["overfit_guard_min_valid_reduction"]
    OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE = guard["overfit_guard_max_valid_after_over_baseline"]
    OVERFIT_GUARD_MAX_VALID_TRAIN_GAP = guard["overfit_guard_max_valid_train_gap"]
    OVERFIT_GUARD_USE_TEST = guard["overfit_guard_use_test"]
    OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE = guard["overfit_guard_max_test_after_over_baseline"]
    overfit_preflight = pd.DataFrame(overfit_recommendation["summary_rows"])
    write_csv(overfit_preflight, OUT_DIR / "overfit_preflight_settings.csv")
    display(overfit_preflight)
    print("RESIDUAL_MODEL_PARAMS =", RESIDUAL_MODEL_PARAMS)

## 4. defect별 bad/good group 로드

**이 셀에서 하는 일**

- defect마다 bad wafer list와 good wafer list를 읽습니다.
- bad/good sample이 겹치지 않는지 검사합니다.
- valid/test split 안에 bad/good sample이 충분히 있는지 warning을 기록합니다.
- 이후 residual boosting에서 group metric을 계산할 수 있도록 `defect_groups` dict를 만듭니다.

**입력**: `DEFECTS`에 적힌 bad/good CSV 파일  
**출력**: `defect_groups`



In [ ]:
defect_groups = {}
for defect in DEFECTS:
    defect_id = defect["defect_id"]
    bad_ids = load_group_ids(defect["bad_group_path"], ID_COL)
    good_ids = load_group_ids(defect["good_group_path"], ID_COL)
    warnings = validate_defect_groups(
        defect_id,
        bad_ids,
        good_ids,
        all_df,
        id_col=ID_COL,
        split_col=SPLIT_COL,
        min_valid_bad_samples=MIN_VALID_BAD_SAMPLES,
    )
    defect_groups[defect_id] = {"bad": bad_ids, "good": good_ids, "warnings": set(warnings)}
    print(defect_id, "bad=", len(bad_ids), "good=", len(good_ids), "warnings=", warnings)

## 5. Baseline CatBoost 학습 및 residual 계산

**이 셀에서 하는 일**

- base feature `Xb`만 사용해 baseline 수율 회귀 모델을 학습합니다.
- 전체 row에 대해 `baseline_pred`를 생성합니다.
- `baseline_residual = yield - baseline_pred`를 계산합니다.
- split별 baseline metric과 defect group별 residual summary를 저장합니다.

**입력**: `train_df`, `valid_df`, `all_df`, `base_feature_cols`  
**출력**: `baseline_model`, `baseline_metrics`, `baseline_summary`, `baseline_pred`, `baseline_residual`



In [ ]:
baseline_model = train_baseline_model(
    train_df,
    valid_df,
    base_feature_cols,
    TARGET_COL,
    BASELINE_MODEL_PARAMS,
)

all_df = add_baseline_predictions(
    baseline_model,
    all_df,
    feature_cols=base_feature_cols,
    target_col=TARGET_COL,
)
train_df, valid_df, test_df = split_frame(all_df, SPLIT_COL)

baseline_metrics = metrics_by_split(all_df, target_col=TARGET_COL, pred_col="baseline_pred", split_col=SPLIT_COL)
baseline_summary = baseline_residual_summary(
    all_df,
    target_col=TARGET_COL,
    pred_col="baseline_pred",
    residual_col="baseline_residual",
    split_col=SPLIT_COL,
    id_col=ID_COL,
    defects=defect_groups,
)

write_csv(baseline_metrics, OUT_DIR / "baseline_metrics.csv")
write_csv(baseline_summary, OUT_DIR / "baseline_residual_summary.csv")

display(baseline_metrics)
display(baseline_summary.head(12))

## 6. defect별 residual feature boosting

**이 셀에서 하는 일**

- defect별로 candidate feature 품질을 먼저 검사합니다.
- 각 round마다 아직 선택되지 않은 candidate feature를 하나씩 residual model에 넣어 평가합니다.
- 선택되는 feature 수는 설정에 따라 1개, 상위 K개, 또는 threshold 통과 feature 전체가 될 수 있습니다.
- residual model은 `Xb`를 쓰지 않고 candidate feature `x_j` 하나만 사용합니다.
- 기본 feature 선택은 validation bad group의 `EXPERIMENT_METRIC` reduction 기준입니다.
- 노트북 설정에서 `top_k` 또는 `threshold` 방식으로 round별 선택 feature 수를 조절할 수 있습니다.
- candidate rank chart의 y축은 기본적으로 `after residual / baseline residual` 비율입니다.
- `ANSWER_FEATURES`에 defect별 정답인자를 넣으면 chart에 별도 마커로 표시됩니다.
- test metric은 선택에 쓰지 않고 기록/검증용으로만 저장합니다.
- round별 ranking, 선택 feature, residual curve를 CSV로 저장합니다.
- `BOOSTING_SHOW_PROGRESS = True`이면 candidate scoring 진행률이 출력됩니다.
- 각 round의 candidate별 after-boosting loss를 rank chart로 보여주고 PNG로 저장합니다.
- 모든 boosting이 끝난 뒤 defect별 round 평균 절대 residual point chart를 보여줍니다.

각 candidate feature는 `x_j -> current residual` 단일 feature 모델로만 평가됩니다. 선택 기준은 validation bad group의 residual 감소입니다.

**입력**: baseline prediction이 포함된 train/valid/test, candidate features, defect groups  
**출력**: `quality_summary`, `selected_features`, `residual_curve`, `round_mean_residual`, `rankings/*.csv`, `plots/*candidate_loss.png`, `plots/round_mean_abs_residual_points.png`



In [ ]:
standard_experiment_start_time = time.perf_counter()

booster = ResidualFeatureBooster(
    ResidualFeatureBoosterConfig(
        residual_model_params=RESIDUAL_MODEL_PARAMS,
        n_rounds=BOOSTING_N_ROUNDS,
        select_per_round=SELECT_PER_ROUND,
        main_metric=BOOSTING_SELECTION_METRIC,
        min_improvement=MIN_IMPROVEMENT,
        selection_mode=BOOSTING_SELECTION_MODE,
        selection_metric=BOOSTING_SELECTION_METRIC,
        selection_threshold=BOOSTING_SELECTION_THRESHOLD,
        selection_direction=BOOSTING_SELECTION_DIRECTION,
        max_select_per_round=BOOSTING_MAX_SELECT_PER_ROUND,
        use_test_for_selection=False,
        min_valid_bad_samples=MIN_VALID_BAD_SAMPLES,
        overfit_guard_enabled=OVERFIT_GUARD_ENABLED,
        overfit_guard_metric_scope=OVERFIT_GUARD_METRIC_SCOPE,
        overfit_guard_metric_name=EXPERIMENT_METRIC_LOWER,
        overfit_guard_min_valid_reduction=OVERFIT_GUARD_MIN_VALID_REDUCTION,
        overfit_guard_max_valid_after_over_baseline=OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE,
        overfit_guard_max_valid_train_gap=OVERFIT_GUARD_MAX_VALID_TRAIN_GAP,
        overfit_guard_use_test=OVERFIT_GUARD_USE_TEST,
        overfit_guard_max_test_after_over_baseline=OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE,
        show_progress=BOOSTING_SHOW_PROGRESS,
        progress_every=BOOSTING_PROGRESS_EVERY,
    )
)

quality_frames = []
selected_frames = []
curve_frames = []
ranking_audit_frames = []
iteration_audit_frames = []
test_prediction_frames = []

# One continuous boosting chain: defect_1 -> defect_2 -> defect_3.
sequential_predictions = {
    "train": train_df["baseline_pred"].to_numpy(dtype=float).copy(),
    "valid": valid_df["baseline_pred"].to_numpy(dtype=float).copy(),
    "test": test_df["baseline_pred"].to_numpy(dtype=float).copy(),
}
global_selected_features = set()
global_iter_offset = 0
defect_initial_predictions = {}

for defect in DEFECTS:
    defect_id = defect["defect_id"]
    groups = defect_groups[defect_id]
    quality = profile_candidate_features(
        all_df,
        candidate_cols,
        id_col=ID_COL,
        split_col=SPLIT_COL,
        bad_ids=groups["bad"],
        good_ids=groups["good"],
        config=FEATURE_FILTER,
        protected_cols={ID_COL, TARGET_COL, SPLIT_COL, *base_feature_cols},
    )
    quality.insert(0, "defect_id", defect_id)
    quality_frames.append(quality)

    if any(str(item).startswith("low_valid_bad_samples") for item in groups.get("warnings", set())):
        print(defect_id, "skipped: low valid bad samples")
        continue

    answer_rules = ANSWER_FEATURES.get(defect_id, [])
    answer_tracking_cols = []
    if answer_rules:
        candidate_series = pd.Series(candidate_cols, dtype=str)
        answer_tracking_cols = candidate_series[answer_feature_mask(candidate_series, answer_rules)].tolist()
    print(f"[{defect_id}] answer tracking cols: {answer_tracking_cols}")
    if answer_rules and not answer_tracking_cols:
        print(f"[WARN {defect_id}] ANSWER_FEATURES did not match any candidate column. Check names/rules: {answer_rules}")

    n_available = sum(str(feature) not in global_selected_features for feature in candidate_cols)
    print(
        f"[{defect_id}] scoring {n_available}/{len(candidate_cols)} candidates; "
        f"inherited selected={len(global_selected_features)}, next global_iter={global_iter_offset + 1}"
    )
    defect_initial_predictions[defect_id] = {
        split: values.copy() for split, values in sequential_predictions.items()
    }
    result = booster.run_for_defect(
        train_df=train_df,
        valid_df=valid_df,
        test_df=test_df,
        candidate_cols=candidate_cols,
        target_col=TARGET_COL,
        id_col=ID_COL,
        baseline_pred_col="baseline_pred",
        defect_id=defect_id,
        bad_sample_ids=groups["bad"],
        good_sample_ids=groups["good"],
        quality_summary=quality,
        initial_predictions=sequential_predictions,
        previously_selected_features=global_selected_features,
        global_iter_start=global_iter_offset,
        base_feature_count=len(base_feature_cols),
        output_dir=OUT_DIR / "rankings",
    )

    # Carry the corrected prediction/residual and selected features into the next defect.
    if result.final_predictions:
        sequential_predictions = {split: values.copy() for split, values in result.final_predictions.items()}
    if not result.selected_features.empty:
        global_selected_features.update(result.selected_features["feature_name"].dropna().astype(str))
    global_iter_offset += len(result.rankings)
    if not result.iteration_summary.empty:
        iteration_audit_frames.append(result.iteration_summary)
    if not result.test_predictions.empty:
        test_prediction_frames.append(result.test_predictions)

    if result.rankings:
        for ranking_df in result.rankings:
            if ranking_df.empty or "round" not in ranking_df.columns:
                continue
            ranking_df = add_answer_feature_flags(ranking_df, feature_col="feature_name", rules=answer_rules)
            ranking_audit_frames.append(ranking_df.copy())
            round_no = int(ranking_df["round"].max())
            answer_mask = ranking_df["is_answer_feature"].fillna(False).astype(bool) if "is_answer_feature" in ranking_df.columns else pd.Series(False, index=ranking_df.index)
            answer_rows = ranking_df[answer_mask].copy()
            if answer_rules and answer_rows.empty:
                print(f"[WARN {defect_id} round {round_no}] no answer feature row in ranking. answer_tracking_cols={answer_tracking_cols}")
            elif not answer_rows.empty:
                answer_cols_to_show = [
                    col for col in [
                        "defect_id", "round", "rank", "feature_name", "selected", "eligible_for_selection",
                        "ranking_only", "already_selected", CANDIDATE_LOSS_BAD_METRIC_COL,
                        CANDIDATE_LOSS_GLOBAL_METRIC_COL, "fail_reason", "overfit_guard_reason",
                        "answer_match_rule",
                    ] if col in answer_rows.columns
                ]
                display(answer_rows[answer_cols_to_show])
            write_csv(ranking_df, OUT_DIR / "rankings" / f"{defect_id}_round_{round_no}.csv")
            fig = plot_candidate_loss_ranking(
                ranking_df,
                output_path=OUT_DIR / "plots" / f"{defect_id}_round_{round_no}_candidate_loss.png",
                global_metric_col=CANDIDATE_LOSS_GLOBAL_METRIC_COL,
                bad_metric_col=CANDIDATE_LOSS_BAD_METRIC_COL,
                answer_features=answer_rules,
                title_prefix=f"{defect_id} round {round_no}",
            )
            if fig is not None:
                display(fig)

    if not result.selected_features.empty:
        selected_for_defect = add_answer_feature_flags(result.selected_features, feature_col="feature_name", rules=answer_rules)
        selected_frames.append(selected_for_defect)
        display(selected_for_defect)
    else:
        print(defect_id, "selected no feature")

    if not result.residual_curve.empty:
        curve_for_defect = add_answer_feature_flags(result.residual_curve, feature_col="selected_feature", rules=answer_rules)
        curve_frames.append(curve_for_defect)

quality_summary = pd.concat(quality_frames, ignore_index=True) if quality_frames else pd.DataFrame()
selected_features = pd.concat(selected_frames, ignore_index=True) if selected_frames else pd.DataFrame()
residual_curve = pd.concat(curve_frames, ignore_index=True) if curve_frames else pd.DataFrame()
boosting_feature_audit = pd.concat(ranking_audit_frames, ignore_index=True) if ranking_audit_frames else pd.DataFrame()
boosting_iteration_audit = pd.concat(iteration_audit_frames, ignore_index=True) if iteration_audit_frames else pd.DataFrame()
boosting_test_predictions = pd.concat(test_prediction_frames, ignore_index=True) if test_prediction_frames else pd.DataFrame()

write_csv(quality_summary, OUT_DIR / "candidate_quality_summary.csv")
write_csv(selected_features, OUT_DIR / "selected_features.csv")
write_csv(residual_curve, OUT_DIR / "residual_reduction_curve.csv")
write_csv(boosting_feature_audit, OUT_DIR / "boosting_feature_metric_audit.csv")
write_csv(boosting_iteration_audit, OUT_DIR / "boosting_iteration_audit.csv")
write_csv(boosting_test_predictions, OUT_DIR / "boosting_test_predictions_by_iteration.csv")
plot_residual_curve(
    residual_curve,
    OUT_DIR,
    answer_features_by_defect=ANSWER_FEATURES,
    metric_name=EXPERIMENT_METRIC_LOWER,
)

# Global iteration summary: iter 0 is baseline; iter 1..N follow the actual defect/round order.
iteration_rows = []
baseline_iter_row = {
    "iter": 0,
    "defect_id": "baseline",
    "round": 0,
    "selected_features": "",
    "n_selected_this_iter": 0,
    "n_base_features": len(base_feature_cols),
    "n_boosted_features": 0,
    "n_total_features": len(base_feature_cols),
}
for split_name, split_frame_df in {"train": train_df, "valid": valid_df, "test": test_df}.items():
    y_true = split_frame_df[TARGET_COL].to_numpy(dtype=float)
    y_pred = split_frame_df["baseline_pred"].to_numpy(dtype=float)
    baseline_iter_row[f"{split_name}_r2"] = r2(y_true, y_pred)
    baseline_iter_row[f"{split_name}_mae"] = mae(y_true, y_pred)
    baseline_iter_row[f"{split_name}_rmse"] = rmse(y_true, y_pred)
iteration_rows.append(baseline_iter_row)

if not boosting_iteration_audit.empty:
    for _, audit_row in boosting_iteration_audit.sort_values("global_iter").iterrows():
        iter_row = {
            "iter": int(audit_row["global_iter"]),
            "defect_id": str(audit_row["defect_id"]),
            "round": int(audit_row["round"]),
            "selected_features": str(audit_row.get("selected_features", "")),
            "n_selected_this_iter": int(audit_row.get("n_selected_this_iter", 0)),
            "n_base_features": int(audit_row.get("n_base_features", len(base_feature_cols))),
            "n_boosted_features": int(audit_row.get("n_cumulative_selected", 0)),
            "n_total_features": int(audit_row.get("n_effective_features", len(base_feature_cols))),
        }
        for split_name in ("train", "valid", "test"):
            iter_row[f"{split_name}_r2"] = audit_row.get(f"{split_name}_global_r2_after", np.nan)
            iter_row[f"{split_name}_mae"] = audit_row.get(f"{split_name}_global_mae_after", np.nan)
            iter_row[f"{split_name}_rmse"] = audit_row.get(f"{split_name}_global_rmse_after", np.nan)
        iteration_rows.append(iter_row)

global_iteration_summary = pd.DataFrame(iteration_rows).sort_values("iter").reset_index(drop=True)
write_csv(global_iteration_summary, OUT_DIR / "global_iteration_feature_metric_summary.csv")

# Save test sample-level predictions/residuals for baseline and every iteration.
test_y = test_df[TARGET_COL].to_numpy(dtype=float)
test_baseline_pred = test_df["baseline_pred"].to_numpy(dtype=float)
test_baseline_raw = pd.DataFrame({
    "iter": 0,
    "global_iter": 0,
    "defect_id": "baseline",
    "round": 0,
    "selected_features": "",
    ID_COL: test_df[ID_COL].astype(str).to_numpy(),
    "y_true": test_y,
    "pred_before": test_baseline_pred,
    "pred_after": test_baseline_pred,
    "residual_before": test_y - test_baseline_pred,
    "residual_after": test_y - test_baseline_pred,
    "abs_residual_before": np.abs(test_y - test_baseline_pred),
    "abs_residual_after": np.abs(test_y - test_baseline_pred),
})
if not boosting_test_predictions.empty:
    test_iter_raw = boosting_test_predictions.copy()
    test_iter_raw["iter"] = pd.to_numeric(test_iter_raw["global_iter"], errors="coerce").astype("Int64")
    test_raw_by_iteration = pd.concat([test_baseline_raw, test_iter_raw], ignore_index=True, sort=False)
else:
    test_raw_by_iteration = test_baseline_raw
write_csv(test_raw_by_iteration, OUT_DIR / "test_raw_predictions_by_iteration.csv")

import matplotlib.pyplot as plt
iter_ticks = global_iteration_summary["iter"].astype(int).tolist()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].plot(global_iteration_summary["iter"], global_iteration_summary["n_base_features"], marker="o", label="base features")
axes[0].plot(global_iteration_summary["iter"], global_iteration_summary["n_boosted_features"], marker="o", label="cumulative boosted features")
axes[0].plot(global_iteration_summary["iter"], global_iteration_summary["n_total_features"], marker="o", label="total effective features")
axes[0].set_title("feature count by global iteration")
axes[0].set_xlabel("iter")
axes[0].set_ylabel("feature count")
axes[0].set_xticks(iter_ticks)
axes[0].grid(alpha=0.25)
axes[0].legend()
axes[1].bar(global_iteration_summary["iter"], global_iteration_summary["n_selected_this_iter"], color="#f28e2b")
axes[1].set_title("features selected in each iteration")
axes[1].set_xlabel("iter")
axes[1].set_ylabel("selected feature count")
axes[1].set_xticks(iter_ticks)
axes[1].grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUT_DIR / "plots" / "global_iteration_feature_counts.png", dpi=150, bbox_inches="tight")
display(fig)

fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
split_colors = {"train": "#59a14f", "valid": "#4e79a7", "test": "#e15759"}
for ax, metric_name in zip(axes, ("mae", "rmse", "r2")):
    for split_name in ("train", "valid", "test"):
        ax.plot(
            global_iteration_summary["iter"],
            global_iteration_summary[f"{split_name}_{metric_name}"],
            marker="o",
            color=split_colors[split_name],
            label=split_name,
        )
    ax.set_title(f"{metric_name.upper()} by global iteration")
    ax.set_xlabel("iter")
    ax.set_ylabel(metric_name.upper())
    ax.set_xticks(iter_ticks)
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "plots" / "global_iteration_train_valid_test_metrics.png", dpi=150, bbox_inches="tight")
display(fig)
display(global_iteration_summary)

display(selected_features)
display(residual_curve)
display(boosting_iteration_audit)
if not boosting_feature_audit.empty:
    highlight_mask = boosting_feature_audit["selected"].fillna(False).astype(bool)
    if "is_answer_feature" in boosting_feature_audit.columns:
        highlight_mask |= boosting_feature_audit["is_answer_feature"].fillna(False).astype(bool)
    display(boosting_feature_audit[highlight_mask])

standard_elapsed_seconds = time.perf_counter() - standard_experiment_start_time
standard_runtime_summary = pd.DataFrame(
    [
        {
            "method": "standard_feature_by_feature",
            "elapsed_seconds": standard_elapsed_seconds,
            "elapsed_minutes": standard_elapsed_seconds / 60.0,
            "n_candidate_features": len(candidate_cols),
            "n_selected_features": len(selected_features),
            "n_defects": len(DEFECTS),
        }
    ]
)
write_csv(standard_runtime_summary, OUT_DIR / "standard_runtime_summary.csv")
display(standard_runtime_summary)


## 6-1. Null Feature Benchmark

real candidate feature와 random/noise feature를 같은 boosting 후보군에 넣고 경쟁시켜, 실제 feature가 noise보다 강하게 선택되는지 확인합니다.

- 기존 6번 결과는 건드리지 않고, 별도 실험으로 `candidate feature + null noise feature`를 다시 boosting합니다.
- PowerSHAP처럼 null feature를 기준선으로 쓰지만, SHAP importance가 아니라 residual improvement metric으로 비교합니다.
- round마다 real feature와 noise feature가 같은 residual 상태에서 같이 경쟁하므로, 후반 round가 noise 수준인지 판단하기 좋습니다.
- 6번과 같은 feature rank curve를 그리고, noise feature 위치는 주황색 X 마커로 표시합니다.
- `null_selected_by_boosting`이 나오거나 top rank에 noise가 자주 들어오면, 해당 round 이후 feature 선택은 약하게 해석해야 합니다.


In [ ]:
NULL_BENCHMARK_ENABLED = True
NULL_BENCHMARK_N_ROUNDS = BOOSTING_N_ROUNDS
NULL_BENCHMARK_N_NOISE_FEATURES = 30
NULL_BENCHMARK_RANDOM_SEED = 2026
NULL_BENCHMARK_TOP_N_DISPLAY = 30
NULL_BENCHMARK_DISPLAY_RANK_CURVES = True
NULL_BENCHMARK_METRIC = BOOSTING_SELECTION_METRIC
NULL_BENCHMARK_USE_TEST_FOR_SELECTION = False
NULL_BENCHMARK_NOISE_PREFIX = "__null_noise_"
NULL_BENCHMARK_DIR = OUT_DIR / "null_benchmark"
NULL_BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)


def _null_metric_name(metric, *, use_test_for_selection=False):
    metric = str(metric)
    if metric in {"bad_rmse_reduction", "rmse_reduction"}:
        return "test_bad_rmse_reduction" if use_test_for_selection else "valid_bad_rmse_reduction"
    if metric.startswith("valid_") or metric.startswith("test_") or metric.startswith("train_"):
        return metric
    return f"valid_{metric}"


def _null_higher_is_better(metric, direction="auto"):
    direction = str(direction).strip().lower()
    if direction in {"higher", "maximize", "max", "gte", ">="}:
        return True
    if direction in {"lower", "minimize", "min", "lte", "<="}:
        return False
    metric = str(metric).lower()
    if "over_baseline" in metric or metric.endswith("_after") or metric.endswith("_ratio"):
        return False
    return True


def _make_null_noise_frame(df, *, n_noise_features, seed, prefix):
    rng = np.random.default_rng(seed)
    noise_data = {
        f"{prefix}{idx:03d}": rng.normal(0.0, 1.0, len(df))
        for idx in range(int(n_noise_features))
    }
    return pd.DataFrame(noise_data, index=df.index)


def _attach_null_noise(df, noise_by_id):
    return df.join(noise_by_id, on=ID_COL)


def _null_adjusted_score(values, *, higher_is_better):
    numeric = pd.to_numeric(values, errors="coerce")
    return numeric if higher_is_better else -numeric


def _feature_type_from_name(series):
    names = series.fillna("").astype(str)
    return np.where(names.str.startswith(NULL_BENCHMARK_NOISE_PREFIX), "null", "real")


def _feature_list_for_defect(frame, *, defect_id, feature_col="feature_name", feature_type=None):
    if frame is None or frame.empty or "defect_id" not in frame.columns or feature_col not in frame.columns:
        return []
    work = frame[frame["defect_id"].astype(str) == str(defect_id)].copy()
    if feature_type is not None and "feature_type" in work.columns:
        work = work[work["feature_type"].astype(str) == str(feature_type)]
    values = work[feature_col].dropna().astype(str).tolist()
    return list(dict.fromkeys(values))


if NULL_BENCHMARK_ENABLED:
    null_metric_col = _null_metric_name(
        NULL_BENCHMARK_METRIC,
        use_test_for_selection=NULL_BENCHMARK_USE_TEST_FOR_SELECTION,
    )
    null_higher_is_better = _null_higher_is_better(null_metric_col, BOOSTING_SELECTION_DIRECTION)
    print("NULL_BENCHMARK_DIR =", NULL_BENCHMARK_DIR)
    print("NULL_BENCHMARK_MODE = real candidates + null noise candidates compete in the same boosting run")
    print("NULL_BENCHMARK_METRIC =", null_metric_col)
    print("NULL_BENCHMARK_N_ROUNDS =", NULL_BENCHMARK_N_ROUNDS)
    print("NULL_BENCHMARK_N_NOISE_FEATURES =", NULL_BENCHMARK_N_NOISE_FEATURES)

    null_noise_frame = _make_null_noise_frame(
        all_df,
        n_noise_features=NULL_BENCHMARK_N_NOISE_FEATURES,
        seed=NULL_BENCHMARK_RANDOM_SEED,
        prefix=NULL_BENCHMARK_NOISE_PREFIX,
    )
    null_feature_cols = list(null_noise_frame.columns)
    null_candidate_cols = list(dict.fromkeys([*candidate_cols, *null_feature_cols]))

    null_all_df = pd.concat([all_df.copy(), null_noise_frame], axis=1)
    noise_by_id = null_all_df.set_index(ID_COL)[null_feature_cols]
    null_train_df = _attach_null_noise(train_df, noise_by_id)
    null_valid_df = _attach_null_noise(valid_df, noise_by_id)
    null_test_df = _attach_null_noise(test_df, noise_by_id)

    null_competition_booster = ResidualFeatureBooster(
        ResidualFeatureBoosterConfig(
            residual_model_params=RESIDUAL_MODEL_PARAMS,
            n_rounds=NULL_BENCHMARK_N_ROUNDS,
            select_per_round=SELECT_PER_ROUND,
            main_metric=NULL_BENCHMARK_METRIC,
            min_improvement=MIN_IMPROVEMENT,
            selection_mode=BOOSTING_SELECTION_MODE,
            selection_metric=NULL_BENCHMARK_METRIC,
            selection_threshold=BOOSTING_SELECTION_THRESHOLD,
            selection_direction=BOOSTING_SELECTION_DIRECTION,
            max_select_per_round=BOOSTING_MAX_SELECT_PER_ROUND,
            use_test_for_selection=NULL_BENCHMARK_USE_TEST_FOR_SELECTION,
            min_valid_bad_samples=MIN_VALID_BAD_SAMPLES,
            overfit_guard_enabled=OVERFIT_GUARD_ENABLED,
            overfit_guard_metric_scope=OVERFIT_GUARD_METRIC_SCOPE,
            overfit_guard_metric_name=EXPERIMENT_METRIC_LOWER,
            overfit_guard_min_valid_reduction=OVERFIT_GUARD_MIN_VALID_REDUCTION,
            overfit_guard_max_valid_after_over_baseline=OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE,
            overfit_guard_max_valid_train_gap=OVERFIT_GUARD_MAX_VALID_TRAIN_GAP,
            overfit_guard_use_test=OVERFIT_GUARD_USE_TEST,
            overfit_guard_max_test_after_over_baseline=OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE,
            show_progress=BOOSTING_SHOW_PROGRESS,
            progress_every=BOOSTING_PROGRESS_EVERY,
        )
    )

    null_competition_summary_rows = []
    null_competition_top_frames = []
    null_competition_ranking_frames = []
    null_competition_selected_frames = []
    null_competition_curve_frames = []
    null_sequential_predictions = {
        "train": null_train_df["baseline_pred"].to_numpy(dtype=float).copy(),
        "valid": null_valid_df["baseline_pred"].to_numpy(dtype=float).copy(),
        "test": null_test_df["baseline_pred"].to_numpy(dtype=float).copy(),
    }
    null_global_selected_features = set()
    null_global_iter_offset = 0

    for defect in DEFECTS:
        defect_id = str(defect["defect_id"])
        groups = defect_groups[defect_id]
        print(f"[NULL COMPETITION {defect_id}] scoring {len(null_candidate_cols)} real+null candidates")

        null_quality = profile_candidate_features(
            null_all_df,
            null_candidate_cols,
            id_col=ID_COL,
            split_col=SPLIT_COL,
            bad_ids=groups["bad"],
            good_ids=groups["good"],
            config=FEATURE_FILTER,
            protected_cols={ID_COL, TARGET_COL, SPLIT_COL, *base_feature_cols},
        )

        result = null_competition_booster.run_for_defect(
            train_df=null_train_df,
            valid_df=null_valid_df,
            test_df=null_test_df,
            candidate_cols=null_candidate_cols,
            target_col=TARGET_COL,
            id_col=ID_COL,
            baseline_pred_col="baseline_pred",
            defect_id=defect_id,
            bad_sample_ids=groups["bad"],
            good_sample_ids=groups["good"],
            quality_summary=null_quality,
            initial_predictions=null_sequential_predictions,
            previously_selected_features=null_global_selected_features,
            global_iter_start=null_global_iter_offset,
            base_feature_count=len(base_feature_cols),
            output_dir=NULL_BENCHMARK_DIR / "competition_rankings",
        )
        if result.final_predictions:
            null_sequential_predictions = {
                split: values.copy() for split, values in result.final_predictions.items()
            }
        if not result.selected_features.empty:
            null_global_selected_features.update(result.selected_features["feature_name"].dropna().astype(str))
        null_global_iter_offset += len(result.rankings)

        if not result.selected_features.empty:
            selected = result.selected_features.copy()
            selected["feature_type"] = _feature_type_from_name(selected["feature_name"])
            selected["is_null_feature"] = selected["feature_type"].eq("null")
            null_competition_selected_frames.append(selected)

        if not result.residual_curve.empty:
            curve = result.residual_curve.copy()
            if "selected_feature" in curve.columns:
                curve["feature_type"] = _feature_type_from_name(curve["selected_feature"])
                curve["is_null_feature"] = curve["feature_type"].eq("null")
            null_competition_curve_frames.append(curve)

        for ranking_df in result.rankings:
            if ranking_df.empty or "round" not in ranking_df.columns:
                continue
            ranking = ranking_df.copy()
            round_no = int(pd.to_numeric(ranking["round"], errors="coerce").max())
            ranking["feature_type"] = _feature_type_from_name(ranking["feature_name"])
            ranking["is_null_feature"] = ranking["feature_type"].eq("null")
            ranking["metric_col"] = null_metric_col
            ranking[null_metric_col] = pd.to_numeric(ranking[null_metric_col], errors="coerce") if null_metric_col in ranking.columns else np.nan

            fail_ok = ranking.get("fail_reason", "").fillna("").astype(str).eq("") if "fail_reason" in ranking.columns else pd.Series(True, index=ranking.index)
            guard_ok = ranking.get("overfit_guard_pass", True).fillna(True).astype(bool) if "overfit_guard_pass" in ranking.columns else pd.Series(True, index=ranking.index)
            selection_ok = ranking.get("eligible_for_selection", True).fillna(True).astype(bool) if "eligible_for_selection" in ranking.columns else pd.Series(True, index=ranking.index)
            metric_ok = ranking[null_metric_col].notna()
            ranking["eligible_for_null_benchmark"] = fail_ok & guard_ok & selection_ok & metric_ok

            top_n = ranking.head(int(NULL_BENCHMARK_TOP_N_DISPLAY)).copy()
            null_competition_top_frames.append(top_n)
            null_competition_ranking_frames.append(ranking)

            if NULL_BENCHMARK_DISPLAY_RANK_CURVES:
                fig = plot_candidate_loss_ranking(
                    ranking,
                    output_path=NULL_BENCHMARK_DIR / "plots" / f"{defect_id}_round_{round_no}_null_competition_rank_curve.png",
                    global_metric_col=CANDIDATE_LOSS_GLOBAL_METRIC_COL,
                    bad_metric_col=CANDIDATE_LOSS_BAD_METRIC_COL,
                    answer_features=ANSWER_FEATURES.get(defect_id, []),
                    title_prefix=f"{defect_id} round {round_no} null competition",
                )
                if fig is not None:
                    display(fig)

            eligible = ranking[ranking["eligible_for_null_benchmark"]].copy()
            real_eligible = eligible[eligible["feature_type"].eq("real")].copy()
            null_eligible = eligible[eligible["feature_type"].eq("null")].copy()
            selected_rows = ranking[ranking.get("selected", False).fillna(False).astype(bool)].copy() if "selected" in ranking.columns else ranking.iloc[0:0].copy()
            selected_real = selected_rows[selected_rows["feature_type"].eq("real")].copy()
            selected_null = selected_rows[selected_rows["feature_type"].eq("null")].copy()

            summary_row = {
                "defect_id": defect_id,
                "round": round_no,
                "metric_col": null_metric_col,
                "higher_is_better": null_higher_is_better,
                "n_real_candidates": int(ranking["feature_type"].eq("real").sum()),
                "n_null_candidates": int(ranking["feature_type"].eq("null").sum()),
                "n_real_eligible": len(real_eligible),
                "n_null_eligible": len(null_eligible),
                "top_n_checked": int(NULL_BENCHMARK_TOP_N_DISPLAY),
                "n_null_in_top_n": int(top_n["feature_type"].eq("null").sum()),
                "n_selected": len(selected_rows),
                "n_selected_real": len(selected_real),
                "n_selected_null": len(selected_null),
                "selected_real_features": ", ".join(selected_real["feature_name"].dropna().astype(str).tolist()),
                "selected_null_features": ", ".join(selected_null["feature_name"].dropna().astype(str).tolist()),
            }

            if real_eligible.empty or null_eligible.empty:
                summary_row["judgement"] = "insufficient_real_or_null_eligible_rows"
                null_competition_summary_rows.append(summary_row)
                continue

            best_real = real_eligible.iloc[0]
            best_null = null_eligible.iloc[0]
            best_real_score = float(best_real[null_metric_col])
            best_null_score = float(best_null[null_metric_col])
            best_real_adjusted = float(_null_adjusted_score(pd.Series([best_real_score]), higher_is_better=null_higher_is_better).iloc[0])
            best_null_adjusted = float(_null_adjusted_score(pd.Series([best_null_score]), higher_is_better=null_higher_is_better).iloc[0])
            null_adjusted = _null_adjusted_score(null_eligible[null_metric_col], higher_is_better=null_higher_is_better)
            real_beats_best_null = bool(best_real_adjusted > best_null_adjusted)
            empirical_p_vs_null = (1 + int(null_adjusted.ge(best_real_adjusted).sum())) / (1 + len(null_adjusted))

            selected_real_adjusted = _null_adjusted_score(selected_real[null_metric_col], higher_is_better=null_higher_is_better) if not selected_real.empty else pd.Series(dtype=float)
            n_selected_real_beating_best_null = int(selected_real_adjusted.gt(best_null_adjusted).sum()) if not selected_real_adjusted.empty else 0

            summary_row.update(
                {
                    "best_real_rank": int(best_real["rank"]),
                    "best_real_feature": best_real["feature_name"],
                    "best_real_score": best_real_score,
                    "best_null_rank": int(best_null["rank"]),
                    "best_null_feature": best_null["feature_name"],
                    "best_null_score": best_null_score,
                    "best_real_margin_vs_best_null": best_real_adjusted - best_null_adjusted,
                    "real_beats_best_null": real_beats_best_null,
                    "n_selected_real_beating_best_null": n_selected_real_beating_best_null,
                    "empirical_p_vs_null": empirical_p_vs_null,
                }
            )

            if len(selected_null) > 0:
                judgement = "null_selected_by_boosting"
            elif real_beats_best_null and int(top_n["feature_type"].eq("null").sum()) == 0:
                judgement = "strong_real_signal_vs_null"
            elif real_beats_best_null:
                judgement = "real_best_beats_null_but_null_appears_in_top_n"
            else:
                judgement = "weak_or_ambiguous_vs_null"
            summary_row["judgement"] = judgement
            null_competition_summary_rows.append(summary_row)

    null_competition_summary = pd.DataFrame(null_competition_summary_rows)
    null_competition_top_ranking = pd.concat(null_competition_top_frames, ignore_index=True) if null_competition_top_frames else pd.DataFrame()
    null_competition_rankings = pd.concat(null_competition_ranking_frames, ignore_index=True) if null_competition_ranking_frames else pd.DataFrame()
    null_competition_selected_features = pd.concat(null_competition_selected_frames, ignore_index=True) if null_competition_selected_frames else pd.DataFrame()
    null_competition_residual_curve = pd.concat(null_competition_curve_frames, ignore_index=True) if null_competition_curve_frames else pd.DataFrame()

    write_csv(null_competition_summary, NULL_BENCHMARK_DIR / "null_competition_summary.csv")
    write_csv(null_competition_top_ranking, NULL_BENCHMARK_DIR / "null_competition_top_ranking.csv")
    write_csv(null_competition_rankings, NULL_BENCHMARK_DIR / "null_competition_rankings.csv")
    write_csv(null_competition_selected_features, NULL_BENCHMARK_DIR / "null_competition_selected_features.csv")
    write_csv(null_competition_residual_curve, NULL_BENCHMARK_DIR / "null_competition_residual_curve.csv")

    comparison_rows = []
    for defect in DEFECTS:
        defect_id = str(defect["defect_id"])
        main_features = _feature_list_for_defect(selected_features, defect_id=defect_id)
        competition_real_features = _feature_list_for_defect(null_competition_selected_features, defect_id=defect_id, feature_type="real")
        competition_null_features = _feature_list_for_defect(null_competition_selected_features, defect_id=defect_id, feature_type="null")
        comparison_rows.append(
            {
                "defect_id": defect_id,
                "main_selected_features": ", ".join(main_features),
                "null_competition_selected_real_features": ", ".join(competition_real_features),
                "null_competition_selected_null_features": ", ".join(competition_null_features),
                "n_main_selected": len(main_features),
                "n_competition_real_selected": len(competition_real_features),
                "n_competition_null_selected": len(competition_null_features),
                "n_overlap_main_vs_competition_real": len(set(main_features) & set(competition_real_features)),
            }
        )
    null_competition_selected_comparison = pd.DataFrame(comparison_rows)
    write_csv(null_competition_selected_comparison, NULL_BENCHMARK_DIR / "null_competition_selected_comparison.csv")

    display(null_competition_summary)
    display(null_competition_selected_comparison)
    display_cols = [
        "defect_id", "round", "rank", "feature_type", "feature_name", "metric_col", null_metric_col,
        "eligible_for_null_benchmark", "selected", "overfit_guard_pass", "fail_reason", "overfit_guard_reason",
    ]
    display(null_competition_top_ranking[[col for col in display_cols if col in null_competition_top_ranking.columns]])

    try:
        import matplotlib.pyplot as plt

        if not null_competition_summary.empty:
            fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
            plot_frame = null_competition_summary.copy()
            for defect_id, group in plot_frame.groupby("defect_id", sort=False):
                group = group.sort_values("round")
                axes[0].plot(group["round"], group["n_null_in_top_n"], marker="o", label=str(defect_id))
                axes[1].plot(group["round"], group["best_real_margin_vs_best_null"], marker="o", label=str(defect_id))
            axes[0].set_title(f"null features in top {NULL_BENCHMARK_TOP_N_DISPLAY}")
            axes[0].set_xlabel("round")
            axes[0].set_ylabel("count")
            axes[0].grid(alpha=0.25)
            axes[0].legend(fontsize=8)
            axes[1].axhline(0.0, color="#6c757d", linestyle="--", linewidth=1.0)
            axes[1].set_title("best real margin vs best null")
            axes[1].set_xlabel("round")
            axes[1].set_ylabel("adjusted metric margin")
            axes[1].grid(alpha=0.25)
            axes[1].legend(fontsize=8)
            fig.tight_layout()
            fig.savefig(NULL_BENCHMARK_DIR / "null_competition_summary_plot.png", dpi=150)
            display(fig)
    except Exception as exc:
        print("null competition plot skipped:", type(exc).__name__, exc)


## 6-2. Feature ranking metric direct check

Feature ranking metric이 올바르게 계산됐는지 대표 feature로 직접 다시 확인합니다.

- 각 defect의 round 1에서 상위, 중간, 하위 feature를 하나씩 고릅니다.
- 같은 residual model을 다시 학습해 설정된 validation loss를 직접 계산합니다.
- ranking CSV의 저장값과 직접 계산값이 같으면 `PASS=True`로 표시합니다.
- round 1은 이전에 선택된 feature가 없어 baseline에서 바로 재현할 수 있으므로 검산 기준으로 사용합니다.


In [ ]:
from feature_boosting.metrics import mae, rmse
from feature_boosting.modeling import fit_regressor, predict_regressor

METRIC_CHECK_ENABLED = True
METRIC_CHECK_ROUND = 1
METRIC_CHECK_TOLERANCE = 1e-7
METRIC_CHECK_NAME = EXPERIMENT_METRIC_LOWER
METRIC_CHECK_FUNCTION = mae if METRIC_CHECK_NAME == "mae" else rmse
METRIC_CHECK_DIR = OUT_DIR / "metric_check"
METRIC_CHECK_DIR.mkdir(parents=True, exist_ok=True)


def _metric_check_representatives(ranking):
    metric_col = f"valid_bad_{METRIC_CHECK_NAME}_after_over_baseline"
    work = ranking.copy()
    if "fail_reason" in work.columns:
        work = work[work["fail_reason"].fillna("").astype(str).eq("")]
    work[metric_col] = pd.to_numeric(work[metric_col], errors="coerce")
    work = work.dropna(subset=[metric_col]).sort_values(metric_col).reset_index(drop=True)
    if work.empty:
        return work

    frames = []
    used = set()
    for label, idx in [("best", 0), ("middle", len(work) // 2), ("worst", len(work) - 1)]:
        feature_name = str(work.iloc[idx]["feature_name"])
        if feature_name in used:
            continue
        used.add(feature_name)
        row = work.iloc[[idx]].copy()
        row["check_position"] = label
        frames.append(row)
    return pd.concat(frames, ignore_index=True) if frames else work.iloc[0:0].copy()


def _metric_values_match(stored, recalculated):
    return bool(np.isclose(
        float(stored),
        float(recalculated),
        rtol=METRIC_CHECK_TOLERANCE,
        atol=METRIC_CHECK_TOLERANCE,
    ))


metric_check_rows = []

if METRIC_CHECK_ENABLED:
    if int(METRIC_CHECK_ROUND) != 1:
        raise ValueError("Direct metric check currently supports round 1 only.")

    y_train_check = train_df[TARGET_COL].to_numpy(dtype=float)
    y_valid_check = valid_df[TARGET_COL].to_numpy(dtype=float)
    baseline_train_check = train_df["baseline_pred"].to_numpy(dtype=float)
    baseline_valid_check = valid_df["baseline_pred"].to_numpy(dtype=float)

    for defect in DEFECTS:
        defect_id = str(defect["defect_id"])
        ranking_path = OUT_DIR / "rankings" / f"{defect_id}_round_{METRIC_CHECK_ROUND}.csv"
        if not ranking_path.exists():
            print(f"[{defect_id}] metric check skipped: ranking file not found")
            continue

        representatives = _metric_check_representatives(pd.read_csv(ranking_path))
        if representatives.empty:
            print(f"[{defect_id}] metric check skipped: no valid ranking rows")
            continue
        if defect_id not in defect_initial_predictions:
            print(f"[{defect_id}] metric check skipped: inherited start prediction not found")
            continue

        inherited_start = defect_initial_predictions[defect_id]
        current_train_check = inherited_start["train"]
        current_valid_check = inherited_start["valid"]
        residual_train_check = y_train_check - current_train_check
        residual_valid_check = y_valid_check - current_valid_check

        bad_mask = valid_df[ID_COL].astype(str).isin(defect_groups[defect_id]["bad"]).to_numpy()
        if not bad_mask.any():
            print(f"[{defect_id}] metric check skipped: no validation bad samples")
            continue

        bad_baseline = METRIC_CHECK_FUNCTION(y_valid_check[bad_mask], baseline_valid_check[bad_mask])
        global_baseline = METRIC_CHECK_FUNCTION(y_valid_check, baseline_valid_check)

        for _, ranking_row in representatives.iterrows():
            feature_name = str(ranking_row["feature_name"])
            model = fit_regressor(
                train_df, residual_train_check, valid_df, residual_valid_check,
                [feature_name], RESIDUAL_MODEL_PARAMS,
            )
            correction = predict_regressor(model, valid_df, [feature_name])
            pred_after = current_valid_check + correction

            direct_bad_after = METRIC_CHECK_FUNCTION(y_valid_check[bad_mask], pred_after[bad_mask])
            direct_bad_ratio = direct_bad_after / bad_baseline
            direct_global_after = METRIC_CHECK_FUNCTION(y_valid_check, pred_after)
            direct_global_ratio = direct_global_after / global_baseline
            stored_bad_after = float(ranking_row[f"valid_bad_{METRIC_CHECK_NAME}_after"])
            stored_bad_ratio = float(ranking_row[f"valid_bad_{METRIC_CHECK_NAME}_after_over_baseline"])
            stored_global_after = float(ranking_row[f"valid_global_{METRIC_CHECK_NAME}_after"])
            stored_global_ratio = float(ranking_row[f"valid_global_{METRIC_CHECK_NAME}_after_over_baseline"])

            checks = [
                _metric_values_match(stored_bad_after, direct_bad_after),
                _metric_values_match(stored_bad_ratio, direct_bad_ratio),
                _metric_values_match(stored_global_after, direct_global_after),
                _metric_values_match(stored_global_ratio, direct_global_ratio),
            ]
            metric_check_rows.append({
                "defect_id": defect_id,
                "global_iter": int(ranking_row.get("global_iter", 0)),
                "round": METRIC_CHECK_ROUND,
                "check_position": ranking_row["check_position"],
                "feature_name": feature_name,
                "metric_name": METRIC_CHECK_NAME,
                "stored_bad_ratio": stored_bad_ratio,
                "direct_bad_ratio": direct_bad_ratio,
                "bad_ratio_abs_diff": abs(stored_bad_ratio - direct_bad_ratio),
                "stored_global_ratio": stored_global_ratio,
                "direct_global_ratio": direct_global_ratio,
                "global_ratio_abs_diff": abs(stored_global_ratio - direct_global_ratio),
                "PASS": all(checks),
            })

metric_check_detail = pd.DataFrame(metric_check_rows)
if not metric_check_detail.empty:
    metric_check_summary = (
        metric_check_detail.groupby("defect_id", as_index=False)
        .agg(
            n_features_checked=("feature_name", "size"),
            n_passed=("PASS", "sum"),
            all_passed=("PASS", "all"),
            max_bad_ratio_abs_diff=("bad_ratio_abs_diff", "max"),
            max_global_ratio_abs_diff=("global_ratio_abs_diff", "max"),
        )
    )
    write_csv(metric_check_detail, METRIC_CHECK_DIR / "metric_check_detail.csv")
    write_csv(metric_check_summary, METRIC_CHECK_DIR / "metric_check_summary.csv")
    display(metric_check_summary)
    display(metric_check_detail)
else:
    metric_check_summary = pd.DataFrame()
    print("No metric check result was created. Run section 6 first.")


## 6-3. Base model + rank 1 features learning curve

각 defect의 round 1에서 rank 1 feature를 가져와 base feature에 추가한 수율 모델을 새로 학습합니다.

- defect별 rank 1 feature를 모으고 중복 feature는 한 번만 사용합니다.
- `base features + rank 1 features`로 CatBoost 수율 모델을 500 iteration까지 학습합니다.
- iteration마다 train, valid, test의 설정 loss와 R2를 계산합니다.
- 최적 iteration은 validation loss만으로 정하고 test는 결과 확인에만 사용합니다.


In [ ]:
from feature_boosting.metrics import mae, r2, rmse

TOP1_CURVE_ENABLED = True
TOP1_CURVE_RANKING_ROUND = 1
TOP1_CURVE_ITERATIONS = 500
TOP1_CURVE_METRIC_NAME = EXPERIMENT_METRIC_LOWER
TOP1_CURVE_METRIC_FUNCTION = mae if TOP1_CURVE_METRIC_NAME == "mae" else rmse
TOP1_CURVE_DIR = OUT_DIR / "base_plus_rank1_curve"
TOP1_CURVE_DIR.mkdir(parents=True, exist_ok=True)

top1_feature_rows = []
top1_feature_cols = []

if TOP1_CURVE_ENABLED:
    for defect in DEFECTS:
        defect_id = str(defect["defect_id"])
        ranking_path = OUT_DIR / "rankings" / f"{defect_id}_round_{TOP1_CURVE_RANKING_ROUND}.csv"
        if not ranking_path.exists():
            print(f"[{defect_id}] rank 1 skipped: ranking file not found")
            continue

        ranking = pd.read_csv(ranking_path)
        ranking["rank"] = pd.to_numeric(ranking["rank"], errors="coerce")
        rank1 = ranking[ranking["rank"].eq(1)].copy()
        if rank1.empty:
            print(f"[{defect_id}] rank 1 skipped: rank 1 row not found")
            continue

        row = rank1.iloc[0]
        feature_name = str(row["feature_name"])
        if feature_name not in candidate_cols:
            print(f"[{defect_id}] rank 1 skipped: {feature_name} is not an available candidate column")
            continue

        top1_feature_rows.append({
            "defect_id": defect_id,
            "round": TOP1_CURVE_RANKING_ROUND,
            "rank": 1,
            "feature_name": feature_name,
            "ranking_metric": row.get("ranking_metric", ""),
            "metric_name": TOP1_CURVE_METRIC_NAME,
            "valid_bad_metric_reduction": row.get(f"valid_bad_{TOP1_CURVE_METRIC_NAME}_reduction", np.nan),
            "valid_bad_metric_after_over_baseline": row.get(f"valid_bad_{TOP1_CURVE_METRIC_NAME}_after_over_baseline", np.nan),
            "selected_in_boosting": bool(row.get("selected", False)),
            "overfit_guard_pass": bool(row.get("overfit_guard_pass", False)),
        })
        if feature_name not in top1_feature_cols:
            top1_feature_cols.append(feature_name)

top1_feature_summary = pd.DataFrame(top1_feature_rows)
write_csv(top1_feature_summary, TOP1_CURVE_DIR / "rank1_features.csv")
display(top1_feature_summary)
print("deduplicated rank 1 features:", top1_feature_cols)

top1_learning_curve = pd.DataFrame()
top1_learning_curve_summary = pd.DataFrame()

if TOP1_CURVE_ENABLED and top1_feature_cols:
    try:
        from catboost import CatBoostRegressor, Pool
    except ImportError as exc:
        raise RuntimeError("Section 6-3 requires CatBoost for staged iteration predictions.") from exc

    curve_feature_cols = list(dict.fromkeys([*base_feature_cols, *top1_feature_cols]))

    def _top1_curve_prepare_frame(df, feature_cols):
        prepared = df[feature_cols].copy()
        for col in prepared.columns:
            if pd.api.types.is_object_dtype(prepared[col]) or isinstance(prepared[col].dtype, pd.CategoricalDtype):
                prepared[col] = prepared[col].astype("object").where(prepared[col].notna(), "__MISSING__")
        return prepared

    train_x_curve = _top1_curve_prepare_frame(train_df, curve_feature_cols)
    valid_x_curve = _top1_curve_prepare_frame(valid_df, curve_feature_cols)
    test_x_curve = _top1_curve_prepare_frame(test_df, curve_feature_cols)
    cat_feature_indices = [
        idx
        for idx, col in enumerate(curve_feature_cols)
        if pd.api.types.is_object_dtype(train_x_curve[col]) or isinstance(train_x_curve[col].dtype, pd.CategoricalDtype)
    ]

    y_train_curve = train_df[TARGET_COL].to_numpy(dtype=float)
    y_valid_curve = valid_df[TARGET_COL].to_numpy(dtype=float)
    y_test_curve = test_df[TARGET_COL].to_numpy(dtype=float)
    train_pool_curve = Pool(train_x_curve, y_train_curve, cat_features=cat_feature_indices)
    valid_pool_curve = Pool(valid_x_curve, y_valid_curve, cat_features=cat_feature_indices)
    test_pool_curve = Pool(test_x_curve, y_test_curve, cat_features=cat_feature_indices)

    curve_model_params = dict(BASELINE_MODEL_PARAMS)
    curve_model_params.pop("backend", None)
    curve_model_params.pop("early_stopping_rounds", None)
    curve_model_params.update({
        "iterations": int(TOP1_CURVE_ITERATIONS),
        "verbose": False,
        "allow_writing_files": False,
    })

    top1_curve_model = CatBoostRegressor(**curve_model_params)
    top1_curve_model.fit(train_pool_curve)

    curve_rows = []
    staged_train = top1_curve_model.staged_predict(train_pool_curve, eval_period=1)
    staged_valid = top1_curve_model.staged_predict(valid_pool_curve, eval_period=1)
    staged_test = top1_curve_model.staged_predict(test_pool_curve, eval_period=1)

    for iteration, (pred_train, pred_valid, pred_test) in enumerate(
        zip(staged_train, staged_valid, staged_test),
        start=1,
    ):
        curve_rows.append({
            "iteration": iteration,
            "metric_name": TOP1_CURVE_METRIC_NAME,
            "train_loss": TOP1_CURVE_METRIC_FUNCTION(y_train_curve, pred_train),
            "valid_loss": TOP1_CURVE_METRIC_FUNCTION(y_valid_curve, pred_valid),
            "test_loss": TOP1_CURVE_METRIC_FUNCTION(y_test_curve, pred_test),
            "train_r2": r2(y_train_curve, pred_train),
            "valid_r2": r2(y_valid_curve, pred_valid),
            "test_r2": r2(y_test_curve, pred_test),
        })

    top1_learning_curve = pd.DataFrame(curve_rows)
    write_csv(top1_learning_curve, TOP1_CURVE_DIR / "base_plus_rank1_learning_curve.csv")

    best_idx = top1_learning_curve["valid_loss"].idxmin()
    best_row = top1_learning_curve.loc[best_idx].copy()
    top1_learning_curve_summary = pd.DataFrame([{
        "n_base_features": len(base_feature_cols),
        "n_rank1_features": len(top1_feature_cols),
        "rank1_features": ", ".join(top1_feature_cols),
        "metric_name": TOP1_CURVE_METRIC_NAME,
        "best_iteration_by_valid_loss": int(best_row["iteration"]),
        "train_loss_at_best": best_row["train_loss"],
        "valid_loss_at_best": best_row["valid_loss"],
        "test_loss_at_best": best_row["test_loss"],
        "train_r2_at_best": best_row["train_r2"],
        "valid_r2_at_best": best_row["valid_r2"],
        "test_r2_at_best": best_row["test_r2"],
    }])
    write_csv(top1_learning_curve_summary, TOP1_CURVE_DIR / "base_plus_rank1_learning_curve_summary.csv")
    display(top1_learning_curve_summary)

    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    colors = {"train": "#2878b5", "valid": "#e07a1f", "test": "#2a9d6f"}
    for split_name in ["train", "valid", "test"]:
        axes[0].plot(
            top1_learning_curve["iteration"],
            top1_learning_curve[f"{split_name}_loss"],
            label=split_name,
            color=colors[split_name],
        )
        axes[1].plot(
            top1_learning_curve["iteration"],
            top1_learning_curve[f"{split_name}_r2"],
            label=split_name,
            color=colors[split_name],
        )

    best_iteration = int(best_row["iteration"])
    for ax in axes:
        ax.axvline(best_iteration, color="#555555", linestyle="--", linewidth=1.2, label=f"best valid iteration = {best_iteration}")
        ax.set_xlabel("CatBoost iteration")
        ax.grid(alpha=0.25)
        ax.legend()

    axes[0].set_title(f"base + rank 1 features: {EXPERIMENT_METRIC} by iteration")
    axes[0].set_ylabel(f"{EXPERIMENT_METRIC} loss")
    axes[1].set_title("base + rank 1 features: R2 by iteration")
    axes[1].set_ylabel("R2")
    fig.tight_layout()
    fig.savefig(TOP1_CURVE_DIR / "base_plus_rank1_loss_r2_curve.png", dpi=150, bbox_inches="tight")
    display(fig)
elif TOP1_CURVE_ENABLED:
    print("Section 6-3 skipped: no rank 1 feature was found. Run section 6 first.")


## 6-4. Boosted/answer feature X-Y data table

각 defect에서 boosting으로 선택된 상위 feature와 `ANSWER_FEATURES`에 입력한 정답인자의 X-Y 데이터를 long-format 표로 만듭니다.

- `defect_id`로 defect_1, defect_2, defect_3을 구분합니다.
- `feature_name`, `x_value`, `y_value`를 한 행에서 확인할 수 있습니다.
- `is_boosted_feature`, `is_answer_feature`로 feature 역할을 구분합니다.
- `defect_group`에는 각 wafer가 bad, good, unlabeled 중 어디에 속하는지 표시합니다.
- `FEATURE_XY_TOP_N_BOOSTED=None`이면 선택된 feature 전체를, `1`이면 defect별 첫 번째 feature만 사용합니다.


In [ ]:
FEATURE_XY_ENABLED = True
FEATURE_XY_TOP_N_BOOSTED = None  # None: all selected features; 1, 5, ...: top N per defect
FEATURE_XY_SAMPLE_SCOPE = "all"  # "all", "bad", or "good"
FEATURE_XY_DIR = OUT_DIR / "feature_xy_tables"
FEATURE_XY_DIR.mkdir(parents=True, exist_ok=True)

feature_xy_catalog_rows = []
feature_xy_frames = []

if FEATURE_XY_ENABLED:
    scope = str(FEATURE_XY_SAMPLE_SCOPE).strip().lower()
    if scope not in {"all", "bad", "good"}:
        raise ValueError("FEATURE_XY_SAMPLE_SCOPE must be 'all', 'bad', or 'good'")

    optional_id_cols = [col for col in ("lot_id", "wf_id") if col in all_df.columns and col != ID_COL]
    base_output_cols = [ID_COL, *optional_id_cols, SPLIT_COL, TARGET_COL]

    for defect in DEFECTS:
        defect_id = str(defect["defect_id"])
        groups = defect_groups[defect_id]

        selected_for_defect = selected_features.iloc[0:0].copy()
        if not selected_features.empty and {"defect_id", "feature_name"}.issubset(selected_features.columns):
            selected_for_defect = selected_features[
                selected_features["defect_id"].astype(str).eq(defect_id)
            ].copy()
            selected_for_defect["_selection_order"] = np.arange(1, len(selected_for_defect) + 1)
            selected_for_defect = selected_for_defect.drop_duplicates("feature_name", keep="first")
            if FEATURE_XY_TOP_N_BOOSTED is not None:
                selected_for_defect = selected_for_defect.head(max(0, int(FEATURE_XY_TOP_N_BOOSTED)))

        boosted_meta = {
            str(row["feature_name"]): {
                "boosting_order": int(row["_selection_order"]),
                "boosting_round": row.get("round", np.nan),
                "boosting_ranking_metric": row.get("ranking_metric", ""),
                "boosting_metric_value": row.get(str(row.get("ranking_metric", "")), np.nan),
            }
            for _, row in selected_for_defect.iterrows()
        }
        boosted_cols = list(boosted_meta)

        answer_rules = ANSWER_FEATURES.get(defect_id, [])
        candidate_series = pd.Series(candidate_cols, dtype=str)
        answer_cols = candidate_series[answer_feature_mask(candidate_series, answer_rules)].tolist() if answer_rules else []
        feature_cols_for_table = list(dict.fromkeys([*boosted_cols, *answer_cols]))

        if not feature_cols_for_table:
            print(f"[{defect_id}] 6-4 skipped: no boosted or answer feature")
            continue

        feature_flags = add_answer_feature_flags(
            pd.DataFrame({"feature_name": feature_cols_for_table}),
            feature_col="feature_name",
            rules=answer_rules,
        ).set_index("feature_name")

        sample_ids = all_df[ID_COL].astype(str)
        defect_group = np.select(
            [sample_ids.isin(groups["bad"]), sample_ids.isin(groups["good"])],
            ["bad", "good"],
            default="unlabeled",
        )

        for feature_name in feature_cols_for_table:
            if feature_name not in all_df.columns:
                print(f"[{defect_id}] 6-4 skipped missing column: {feature_name}")
                continue

            meta = boosted_meta.get(feature_name, {})
            is_boosted = feature_name in boosted_meta
            is_answer = bool(feature_flags.loc[feature_name, "is_answer_feature"])
            feature_role = "boosted_and_answer" if is_boosted and is_answer else ("boosted" if is_boosted else "answer")
            answer_rule = feature_flags.loc[feature_name, "answer_match_rule"] if "answer_match_rule" in feature_flags.columns else ""
            catalog_row = {
                "defect_id": defect_id,
                "feature_name": feature_name,
                "feature_role": feature_role,
                "is_boosted_feature": is_boosted,
                "is_answer_feature": is_answer,
                "boosting_order": meta.get("boosting_order", np.nan),
                "boosting_round": meta.get("boosting_round", np.nan),
                "boosting_ranking_metric": meta.get("boosting_ranking_metric", ""),
                "boosting_metric_value": meta.get("boosting_metric_value", np.nan),
                "answer_match_rule": answer_rule,
            }
            feature_xy_catalog_rows.append(catalog_row)

            xy = all_df[base_output_cols].copy()
            xy.insert(0, "defect_id", defect_id)
            xy["defect_group"] = defect_group
            xy["feature_name"] = feature_name
            xy["feature_role"] = catalog_row["feature_role"]
            xy["x_value"] = all_df[feature_name].to_numpy()
            xy["y_value"] = xy[TARGET_COL]
            xy["is_boosted_feature"] = catalog_row["is_boosted_feature"]
            xy["is_answer_feature"] = catalog_row["is_answer_feature"]
            xy["boosting_order"] = catalog_row["boosting_order"]
            xy["boosting_round"] = catalog_row["boosting_round"]
            xy["answer_match_rule"] = catalog_row["answer_match_rule"]
            xy = xy.drop(columns=[TARGET_COL])
            if scope != "all":
                xy = xy[xy["defect_group"].eq(scope)].copy()
            feature_xy_frames.append(xy)

feature_xy_catalog = pd.DataFrame(feature_xy_catalog_rows)
feature_xy_data = pd.concat(feature_xy_frames, ignore_index=True) if feature_xy_frames else pd.DataFrame()
write_csv(feature_xy_catalog, FEATURE_XY_DIR / "boosted_answer_feature_catalog.csv")
write_csv(feature_xy_data, FEATURE_XY_DIR / "boosted_answer_xy_long.csv")

print("feature catalog rows:", len(feature_xy_catalog))
print("X-Y data rows:", len(feature_xy_data))
display(feature_xy_catalog)
display(feature_xy_data.head(30))


## 7. Final model: Xb + selected Xnew 재학습

**이 셀에서 하는 일**

- residual boosting에서 선택된 feature 목록을 중복 없이 정리합니다.
- 최종 feature set을 `base_feature_cols + selected_cols`로 만듭니다.
- final CatBoost 회귀 모델을 새로 학습합니다.
- baseline 모델과 final 모델의 train/valid/test 및 defect bad/good group 성능을 비교합니다.
- final feature 구성을 막대 차트로 보여줍니다.
- baseline vs final 성능을 설정된 `EXPERIMENT_METRIC` 기준으로 비교합니다.

**입력**: `selected_features`, `base_feature_cols`, `all_df`  
**출력**: `final_model`, `final_pred`, `final_model_metrics.csv`, `final_model_metric_summary.csv`, `plots/final_model_metric_comparison.png`



In [ ]:
selected_cols = []
if not selected_features.empty:
    for feature in selected_features["feature_name"].dropna().astype(str):
        if feature in candidate_cols and feature not in selected_cols:
            selected_cols.append(feature)

final_feature_cols = base_feature_cols + selected_cols
final_feature_summary = pd.DataFrame(
    [
        {"feature_type": "base", "count": len(base_feature_cols)},
        {"feature_type": "selected", "count": len(selected_cols)},
    ]
)
write_csv(final_feature_summary, OUT_DIR / "final_feature_set_summary.csv")

print("selected_cols:", selected_cols)
print("final feature count:", len(final_feature_cols))
display(final_feature_summary)
fig = plot_final_feature_set_summary(
    final_feature_summary,
    output_path=OUT_DIR / "plots" / "final_feature_set_summary.png",
)
if fig is not None:
    display(fig)

final_model = train_final_model(
    train_df,
    valid_df,
    feature_cols=final_feature_cols,
    target_col=TARGET_COL,
    catboost_params=FINAL_MODEL_PARAMS,
)
all_df["final_pred"] = predict_final(final_model, all_df, final_feature_cols)

final_metrics = pd.concat(
    [
        evaluate_model_by_groups(
            all_df,
            model_name="baseline",
            pred_col="baseline_pred",
            target_col=TARGET_COL,
            split_col=SPLIT_COL,
            id_col=ID_COL,
            defects=defect_groups,
        ),
        evaluate_model_by_groups(
            all_df,
            model_name="final",
            pred_col="final_pred",
            target_col=TARGET_COL,
            split_col=SPLIT_COL,
            id_col=ID_COL,
            defects=defect_groups,
        ),
    ],
    ignore_index=True,
)
write_csv(final_metrics, OUT_DIR / "final_model_metrics.csv")

final_metric_summary_df = final_metric_summary(final_metrics)
write_csv(final_metric_summary_df, OUT_DIR / "final_model_metric_summary.csv")
fig = plot_final_metric_comparison(
    final_metric_summary_df,
    output_path=OUT_DIR / "plots" / "final_model_metric_comparison.png",
    metrics=(EXPERIMENT_METRIC_LOWER,),
)
if fig is not None:
    display(fig)

display(final_metric_summary_df)
display(final_metrics)


## 8. Optional SHAP 검증

**이 셀에서 하는 일**

- final model 기준으로 SHAP summary를 계산합니다.
- 전체 test sample, defect별 bad group, defect별 good group을 나누어 feature importance를 기록합니다.
- residual boosting으로 선택된 feature가 final model에서도 상위권에 나타나는지 확인할 수 있게 합니다.
- CatBoost/SHAP 계산이 불가능한 환경에서는 skip 상태를 CSV에 남깁니다.

SHAP은 원인 확정이 아니라, 선택 feature가 final model에서 실제로 사용되는지 확인하는 보조 검증입니다.

**입력**: `final_model`, test set, `selected_features`  
**출력**: `shap_summary.csv`



In [ ]:
if SHAP_ENABLED:
    shap_summary = compute_shap_summary(
        final_model,
        all_df[all_df[SPLIT_COL].astype(str) == "test"],
        feature_cols=final_feature_cols,
        id_col=ID_COL,
        defects=defect_groups,
        selected_features=selected_features,
        max_samples=SHAP_MAX_SAMPLES,
    )
else:
    shap_summary = pd.DataFrame([{"status": "disabled"}])

write_csv(shap_summary, OUT_DIR / "shap_summary.csv")
display(shap_summary.head(30))

## 9. 산출물 확인

**이 셀에서 하는 일**

- 이번 run의 output directory를 출력합니다.
- 저장된 CSV, model, plot 파일 목록을 보여줍니다.
- 노트북 실행 후 어떤 결과물이 생겼는지 빠르게 확인하는 마지막 점검 셀입니다.

**입력**: `OUT_DIR`  
**출력**: 저장된 산출물 목록



In [ ]:
print("saved to:", OUT_DIR)
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUT_DIR))

## 10. Second experiment: block preselection residual boosting

**? ??? ?? ?**

- ?? feature-by-feature boosting ??? ??? ???.
- candidate feature? `BLOCK_SIZE`?? ?? block residual model? ?? ?????.
- stage 1?? ?? ?? block 1?? ????.
- stage 2?? ??? block ?? feature? 1?? ?? ??/?????.
- ??? feature ??? ???? `top_k` ?? `threshold` ???? ?????.

**??**: `OUT_DIR / "block_experiment"` ?? block ranking, block ?? feature ranking, ?? ?? feature CSV? plot


In [ ]:
from feature_boosting.metrics import mae, rmse, reduction
from feature_boosting.modeling import fit_regressor, predict_regressor

BLOCK_EXPERIMENT_ENABLED = True
BLOCK_SIZE = 10
BLOCK_STAGE1_SELECT_BLOCKS = 1
BLOCK_STAGE1_SELECTION_METRIC = BOOSTING_SELECTION_METRIC
BLOCK_FINAL_SELECTION_MODE = BOOSTING_SELECTION_MODE  # "top_k" or "threshold"
BLOCK_FINAL_SELECTION_METRIC = BOOSTING_SELECTION_METRIC
BLOCK_FINAL_TOP_K = SELECT_PER_ROUND
BLOCK_FINAL_THRESHOLD = BOOSTING_SELECTION_THRESHOLD
BLOCK_EXPERIMENT_DIR = OUT_DIR / "block_experiment"
BLOCK_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print("BLOCK_EXPERIMENT_DIR =", BLOCK_EXPERIMENT_DIR)
print("BLOCK_SIZE =", BLOCK_SIZE)
print("BLOCK_FINAL_SELECTION_MODE =", BLOCK_FINAL_SELECTION_MODE)


In [ ]:
def _block_metric_name(metric, *, use_test=False):
    if metric in {"bad_rmse_reduction", "rmse_reduction"}:
        return "test_bad_rmse_reduction" if use_test else "valid_bad_rmse_reduction"
    if metric.startswith("valid_") or metric.startswith("test_") or metric.startswith("train_"):
        return metric
    return f"valid_{metric}"


def _block_higher_is_better(metric):
    lower_metric = str(metric).lower()
    if "over_baseline" in lower_metric or lower_metric.endswith("_after") or lower_metric.endswith("_ratio"):
        return False
    return True


def _safe_ratio(numerator, denominator):
    if not np.isfinite(numerator) or not np.isfinite(denominator) or denominator == 0:
        return float("nan")
    return float(numerator / denominator)


def _safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


def _mask_ids(df, sample_ids):
    return df[ID_COL].astype(str).isin(sample_ids).to_numpy()


def _add_block_split_metrics(row, *, prefix, df, y, before_pred, after_pred, baseline_pred, groups):
    masks = {
        "bad": _mask_ids(df, groups["bad"]),
        "good": _mask_ids(df, groups["good"]),
        "global": np.ones(len(df), dtype=bool),
    }
    for group_name, mask in masks.items():
        before_rmse = rmse(y[mask], before_pred[mask])
        after_rmse = rmse(y[mask], after_pred[mask])
        baseline_rmse = rmse(y[mask], baseline_pred[mask])
        before_mae = mae(y[mask], before_pred[mask])
        after_mae = mae(y[mask], after_pred[mask])
        baseline_mae = mae(y[mask], baseline_pred[mask])
        row[f"{prefix}_{group_name}_rmse_before"] = before_rmse
        row[f"{prefix}_{group_name}_rmse_after"] = after_rmse
        row[f"{prefix}_{group_name}_rmse_reduction"] = reduction(before_rmse, after_rmse)
        row[f"{prefix}_{group_name}_rmse_baseline"] = baseline_rmse
        row[f"{prefix}_{group_name}_rmse_after_over_baseline"] = _safe_ratio(after_rmse, baseline_rmse)
        row[f"{prefix}_{group_name}_mae_before"] = before_mae
        row[f"{prefix}_{group_name}_mae_after"] = after_mae
        row[f"{prefix}_{group_name}_mae_reduction"] = reduction(before_mae, after_mae)
        row[f"{prefix}_{group_name}_mae_baseline"] = baseline_mae
        row[f"{prefix}_{group_name}_mae_after_over_baseline"] = _safe_ratio(after_mae, baseline_mae)


def _apply_block_overfit_guard(row):
    scope = OVERFIT_GUARD_METRIC_SCOPE
    row["overfit_guard_scope"] = scope
    row["overfit_guard_metric"] = EXPERIMENT_METRIC_LOWER
    if row.get("fail_reason"):
        row["overfit_guard_pass"] = False
        row["overfit_guard_reason"] = f"not_evaluated:{row.get('fail_reason')}"
        row["overfit_gap_valid_train_metric_ratio"] = np.nan
        return row
    if not OVERFIT_GUARD_ENABLED:
        row["overfit_guard_pass"] = True
        row["overfit_guard_reason"] = ""
        return row

    reasons = []
    valid_reduction_col = f"valid_{scope}_{EXPERIMENT_METRIC_LOWER}_reduction"
    valid_ratio_col = f"valid_{scope}_{EXPERIMENT_METRIC_LOWER}_after_over_baseline"
    train_ratio_col = f"train_{scope}_{EXPERIMENT_METRIC_LOWER}_after_over_baseline"
    test_ratio_col = f"test_{scope}_{EXPERIMENT_METRIC_LOWER}_after_over_baseline"

    valid_reduction = _safe_float(row.get(valid_reduction_col))
    if OVERFIT_GUARD_MIN_VALID_REDUCTION is not None and (
        not np.isfinite(valid_reduction) or valid_reduction <= float(OVERFIT_GUARD_MIN_VALID_REDUCTION)
    ):
        reasons.append(f"{valid_reduction_col}<={float(OVERFIT_GUARD_MIN_VALID_REDUCTION):g}")

    valid_ratio = _safe_float(row.get(valid_ratio_col))
    if OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE is not None and (
        not np.isfinite(valid_ratio) or valid_ratio > float(OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE)
    ):
        reasons.append(f"{valid_ratio_col}>{float(OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE):g}")

    train_ratio = _safe_float(row.get(train_ratio_col))
    gap = valid_ratio - train_ratio if np.isfinite(valid_ratio) and np.isfinite(train_ratio) else float("nan")
    row["overfit_gap_valid_train_metric_ratio"] = gap
    if OVERFIT_GUARD_MAX_VALID_TRAIN_GAP is not None and np.isfinite(gap) and gap > float(OVERFIT_GUARD_MAX_VALID_TRAIN_GAP):
        reasons.append(f"valid_train_{scope}_{EXPERIMENT_METRIC_LOWER}_ratio_gap>{float(OVERFIT_GUARD_MAX_VALID_TRAIN_GAP):g}")

    if OVERFIT_GUARD_USE_TEST:
        test_ratio = _safe_float(row.get(test_ratio_col))
        if OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE is not None and np.isfinite(test_ratio) and test_ratio > float(OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE):
            reasons.append(f"{test_ratio_col}>{float(OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE):g}")

    row["overfit_guard_pass"] = not reasons
    row["overfit_guard_reason"] = ";".join(reasons)
    return row


def _score_residual_feature_set(feature_cols, *, defect_id, groups, stage, feature_set_name):
    row = {
        "defect_id": defect_id,
        "stage": stage,
        "feature_set_name": feature_set_name,
        "feature_name": feature_set_name,
        "n_features": len(feature_cols),
        "feature_names": "|".join(feature_cols),
        "fail_reason": "",
    }
    missing = [col for col in feature_cols if col not in train_df.columns]
    if missing:
        row["fail_reason"] = f"missing_column:{missing[:3]}"
        return _apply_block_overfit_guard(row)

    y_train = train_df[TARGET_COL].to_numpy(dtype=float)
    y_valid = valid_df[TARGET_COL].to_numpy(dtype=float)
    y_test = test_df[TARGET_COL].to_numpy(dtype=float)
    base_train = train_df["baseline_pred"].to_numpy(dtype=float)
    base_valid = valid_df["baseline_pred"].to_numpy(dtype=float)
    base_test = test_df["baseline_pred"].to_numpy(dtype=float)
    residual_train = y_train - base_train
    residual_valid = y_valid - base_valid

    try:
        model = fit_regressor(train_df, residual_train, valid_df, residual_valid, feature_cols, RESIDUAL_MODEL_PARAMS)
        pred_train_delta = predict_regressor(model, train_df, feature_cols)
        pred_valid_delta = predict_regressor(model, valid_df, feature_cols)
        pred_test_delta = predict_regressor(model, test_df, feature_cols)
    except Exception as exc:
        row["fail_reason"] = f"model_fit_failed:{type(exc).__name__}"
        return _apply_block_overfit_guard(row)

    _add_block_split_metrics(
        row,
        prefix="train",
        df=train_df,
        y=y_train,
        before_pred=base_train,
        after_pred=base_train + pred_train_delta,
        baseline_pred=base_train,
        groups=groups,
    )
    _add_block_split_metrics(
        row,
        prefix="valid",
        df=valid_df,
        y=y_valid,
        before_pred=base_valid,
        after_pred=base_valid + pred_valid_delta,
        baseline_pred=base_valid,
        groups=groups,
    )
    _add_block_split_metrics(
        row,
        prefix="test",
        df=test_df,
        y=y_test,
        before_pred=base_test,
        after_pred=base_test + pred_test_delta,
        baseline_pred=base_test,
        groups=groups,
    )
    return _apply_block_overfit_guard(row)


def _rank_block_rows(frame, *, metric_col, higher_is_better):
    if frame.empty:
        return frame
    result = frame.copy()
    result[metric_col] = pd.to_numeric(result[metric_col], errors="coerce") if metric_col in result.columns else np.nan
    result = result.sort_values([metric_col, "feature_set_name"], ascending=[not higher_is_better, True], na_position="last").reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result.insert(1, "ranking_metric", metric_col)
    return result


def _select_from_ranked_frame(frame, *, metric_col, mode, top_k, threshold, higher_is_better):
    if frame.empty or metric_col not in frame.columns:
        return frame.iloc[0:0].copy()
    eligible = frame[
        frame["fail_reason"].fillna("").astype(str).eq("")
        & frame["overfit_guard_pass"].fillna(False).astype(bool)
        & pd.to_numeric(frame[metric_col], errors="coerce").notna()
    ].copy()
    if eligible.empty:
        return eligible
    mode = str(mode).strip().lower()
    if mode in {"threshold", "metric_threshold", "ratio_threshold"}:
        if threshold is None:
            print(f"[BLOCK EXPERIMENT] threshold mode selected, but threshold is None. No feature selected.")
            return eligible.iloc[0:0].copy()
        values = pd.to_numeric(eligible[metric_col], errors="coerce")
        keep = values >= float(threshold) if higher_is_better else values <= float(threshold)
        return eligible[keep].copy()
    selected = eligible.head(max(1, int(top_k))).copy()
    if higher_is_better:
        selected = selected[pd.to_numeric(selected[metric_col], errors="coerce") > MIN_IMPROVEMENT]
    elif threshold is not None:
        selected = selected[pd.to_numeric(selected[metric_col], errors="coerce") <= float(threshold)]
    return selected


In [ ]:
if BLOCK_EXPERIMENT_ENABLED:
    standard_one_round_start_time = time.perf_counter()
    standard_one_round_booster = ResidualFeatureBooster(
        ResidualFeatureBoosterConfig(
            residual_model_params=RESIDUAL_MODEL_PARAMS,
            n_rounds=1,
            select_per_round=SELECT_PER_ROUND,
            main_metric=BOOSTING_SELECTION_METRIC,
            min_improvement=MIN_IMPROVEMENT,
            selection_mode=BOOSTING_SELECTION_MODE,
            selection_metric=BOOSTING_SELECTION_METRIC,
            selection_threshold=BOOSTING_SELECTION_THRESHOLD,
            selection_direction=BOOSTING_SELECTION_DIRECTION,
            max_select_per_round=BOOSTING_MAX_SELECT_PER_ROUND,
            use_test_for_selection=False,
            min_valid_bad_samples=MIN_VALID_BAD_SAMPLES,
            overfit_guard_enabled=OVERFIT_GUARD_ENABLED,
            overfit_guard_metric_scope=OVERFIT_GUARD_METRIC_SCOPE,
            overfit_guard_metric_name=EXPERIMENT_METRIC_LOWER,
            overfit_guard_min_valid_reduction=OVERFIT_GUARD_MIN_VALID_REDUCTION,
            overfit_guard_max_valid_after_over_baseline=OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE,
            overfit_guard_max_valid_train_gap=OVERFIT_GUARD_MAX_VALID_TRAIN_GAP,
            overfit_guard_use_test=OVERFIT_GUARD_USE_TEST,
            overfit_guard_max_test_after_over_baseline=OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE,
            show_progress=BOOSTING_SHOW_PROGRESS,
            progress_every=BOOSTING_PROGRESS_EVERY,
        )
    )
    standard_one_round_quality_frames = []
    standard_one_round_ranking_frames = []
    standard_one_round_selected_frames = []
    standard_one_round_curve_frames = []

    for defect in DEFECTS:
        defect_id = defect["defect_id"]
        groups = defect_groups[defect_id]
        answer_rules = ANSWER_FEATURES.get(defect_id, [])
        quality = profile_candidate_features(
            all_df,
            candidate_cols,
            id_col=ID_COL,
            split_col=SPLIT_COL,
            bad_ids=groups["bad"],
            good_ids=groups["good"],
            config=FEATURE_FILTER,
            protected_cols={ID_COL, TARGET_COL, SPLIT_COL, *base_feature_cols},
        )
        quality.insert(0, "defect_id", defect_id)
        standard_one_round_quality_frames.append(quality)
        if any(str(item).startswith("low_valid_bad_samples") for item in groups.get("warnings", set())):
            print(f"[{defect_id} standard one-round] skipped: low valid bad samples")
            continue
        print(f"[{defect_id} standard one-round] scoring {len(candidate_cols)} candidates")
        result = standard_one_round_booster.run_for_defect(
            train_df=train_df,
            valid_df=valid_df,
            test_df=test_df,
            candidate_cols=candidate_cols,
            target_col=TARGET_COL,
            id_col=ID_COL,
            baseline_pred_col="baseline_pred",
            defect_id=defect_id,
            bad_sample_ids=groups["bad"],
            good_sample_ids=groups["good"],
            quality_summary=quality,
            output_dir=BLOCK_EXPERIMENT_DIR / "standard_one_round_rankings",
        )
        standard_one_round_ranking_frames.extend(result.rankings)
        if not result.selected_features.empty:
            standard_one_round_selected_frames.append(add_answer_feature_flags(result.selected_features, feature_col="feature_name", rules=answer_rules))
        if not result.residual_curve.empty:
            standard_one_round_curve_frames.append(add_answer_feature_flags(result.residual_curve, feature_col="selected_feature", rules=answer_rules))

    standard_one_round_elapsed_seconds = time.perf_counter() - standard_one_round_start_time
    standard_one_round_quality_summary = pd.concat(standard_one_round_quality_frames, ignore_index=True) if standard_one_round_quality_frames else pd.DataFrame()
    standard_one_round_rankings = pd.concat(standard_one_round_ranking_frames, ignore_index=True) if standard_one_round_ranking_frames else pd.DataFrame()
    standard_one_round_selected_features = pd.concat(standard_one_round_selected_frames, ignore_index=True) if standard_one_round_selected_frames else pd.DataFrame()
    standard_one_round_residual_curve = pd.concat(standard_one_round_curve_frames, ignore_index=True) if standard_one_round_curve_frames else pd.DataFrame()
    write_csv(standard_one_round_quality_summary, BLOCK_EXPERIMENT_DIR / "standard_one_round_candidate_quality_summary.csv")
    write_csv(standard_one_round_rankings, BLOCK_EXPERIMENT_DIR / "standard_one_round_rankings.csv")
    write_csv(standard_one_round_selected_features, BLOCK_EXPERIMENT_DIR / "standard_one_round_selected_features.csv")
    write_csv(standard_one_round_residual_curve, BLOCK_EXPERIMENT_DIR / "standard_one_round_residual_curve.csv")

    block_experiment_start_time = time.perf_counter()
    block_quality_frames = []
    block_ranking_frames = []
    block_feature_ranking_frames = []
    block_selected_feature_frames = []

    block_metric_col = _block_metric_name(BLOCK_STAGE1_SELECTION_METRIC, use_test=False)
    block_higher_is_better = _block_higher_is_better(block_metric_col)
    feature_metric_col = _block_metric_name(BLOCK_FINAL_SELECTION_METRIC, use_test=False)
    feature_higher_is_better = _block_higher_is_better(feature_metric_col)

    for defect in DEFECTS:
        defect_id = defect["defect_id"]
        groups = defect_groups[defect_id]
        answer_rules = ANSWER_FEATURES.get(defect_id, [])

        quality = profile_candidate_features(
            all_df,
            candidate_cols,
            id_col=ID_COL,
            split_col=SPLIT_COL,
            bad_ids=groups["bad"],
            good_ids=groups["good"],
            config=FEATURE_FILTER,
            protected_cols={ID_COL, TARGET_COL, SPLIT_COL, *base_feature_cols},
        )
        quality.insert(0, "defect_id", defect_id)
        block_quality_frames.append(quality)
        passing_candidates = quality.loc[quality["is_pass"].astype(bool), "feature_name"].astype(str).tolist()
        blocks = [passing_candidates[idx : idx + BLOCK_SIZE] for idx in range(0, len(passing_candidates), BLOCK_SIZE)]
        print(f"[{defect_id} block experiment] passing candidates={len(passing_candidates)}, blocks={len(blocks)}")
        if not blocks:
            continue

        block_rows = []
        for block_idx, block_cols in enumerate(blocks, start=1):
            block_name = f"block_{block_idx:04d}"
            row = _score_residual_feature_set(block_cols, defect_id=defect_id, groups=groups, stage="block", feature_set_name=block_name)
            row["block_id"] = block_name
            row["block_start_feature"] = block_cols[0] if block_cols else ""
            row["block_end_feature"] = block_cols[-1] if block_cols else ""
            row["contains_answer_feature"] = bool(answer_rules and answer_feature_mask(pd.Series(block_cols, dtype=str), answer_rules).any())
            block_rows.append(row)
            if block_idx == 1 or block_idx == len(blocks) or block_idx % 10 == 0:
                print(f"[{defect_id} block experiment] scored block {block_idx}/{len(blocks)}")

        block_ranking = _rank_block_rows(pd.DataFrame(block_rows), metric_col=block_metric_col, higher_is_better=block_higher_is_better)
        block_ranking["selected_block"] = False
        selected_blocks = _select_from_ranked_frame(
            block_ranking,
            metric_col=block_metric_col,
            mode="top_k",
            top_k=BLOCK_STAGE1_SELECT_BLOCKS,
            threshold=None,
            higher_is_better=block_higher_is_better,
        )
        selected_block_ids = set(selected_blocks.get("block_id", pd.Series(dtype=str)).astype(str))
        block_ranking["selected_block"] = block_ranking["block_id"].astype(str).isin(selected_block_ids)
        write_csv(block_ranking, BLOCK_EXPERIMENT_DIR / f"{defect_id}_block_ranking.csv")
        block_ranking_frames.append(block_ranking)

        if selected_blocks.empty:
            print(f"[{defect_id} block experiment] selected no block")
            continue

        selected_block = selected_blocks.iloc[0]
        selected_block_id = str(selected_block["block_id"])
        selected_block_features = str(selected_block["feature_names"]).split("|")
        print(f"[{defect_id} block experiment] selected block: {selected_block_id}, n_features={len(selected_block_features)}")

        feature_rows = []
        for feature in selected_block_features:
            row = _score_residual_feature_set([feature], defect_id=defect_id, groups=groups, stage="feature_in_selected_block", feature_set_name=feature)
            row["block_id"] = selected_block_id
            row["feature_name"] = feature
            feature_rows.append(row)
        feature_ranking = _rank_block_rows(pd.DataFrame(feature_rows), metric_col=feature_metric_col, higher_is_better=feature_higher_is_better)
        feature_ranking = add_answer_feature_flags(feature_ranking, feature_col="feature_name", rules=answer_rules)
        selected_feature_rows = _select_from_ranked_frame(
            feature_ranking,
            metric_col=feature_metric_col,
            mode=BLOCK_FINAL_SELECTION_MODE,
            top_k=BLOCK_FINAL_TOP_K,
            threshold=BLOCK_FINAL_THRESHOLD,
            higher_is_better=feature_higher_is_better,
        )
        selected_feature_names = set(selected_feature_rows.get("feature_name", pd.Series(dtype=str)).astype(str))
        feature_ranking["selected"] = feature_ranking["feature_name"].astype(str).isin(selected_feature_names)
        write_csv(feature_ranking, BLOCK_EXPERIMENT_DIR / f"{defect_id}_selected_block_feature_ranking.csv")
        block_feature_ranking_frames.append(feature_ranking)

        if not selected_feature_rows.empty:
            selected_feature_rows = selected_feature_rows.copy()
            selected_feature_rows["defect_id"] = defect_id
            selected_feature_rows["block_id"] = selected_block_id
            selected_feature_rows["selection_mode"] = BLOCK_FINAL_SELECTION_MODE
            block_selected_feature_frames.append(selected_feature_rows)
        else:
            print(f"[{defect_id} block experiment] selected no feature inside {selected_block_id}")

    block_quality_summary = pd.concat(block_quality_frames, ignore_index=True) if block_quality_frames else pd.DataFrame()
    block_rankings = pd.concat(block_ranking_frames, ignore_index=True) if block_ranking_frames else pd.DataFrame()
    block_feature_rankings = pd.concat(block_feature_ranking_frames, ignore_index=True) if block_feature_ranking_frames else pd.DataFrame()
    block_selected_features = pd.concat(block_selected_feature_frames, ignore_index=True) if block_selected_feature_frames else pd.DataFrame()

    write_csv(block_quality_summary, BLOCK_EXPERIMENT_DIR / "block_candidate_quality_summary.csv")
    write_csv(block_rankings, BLOCK_EXPERIMENT_DIR / "block_rankings.csv")
    write_csv(block_feature_rankings, BLOCK_EXPERIMENT_DIR / "selected_block_feature_rankings.csv")
    write_csv(block_selected_features, BLOCK_EXPERIMENT_DIR / "block_experiment_selected_features.csv")

    print("block experiment saved to:", BLOCK_EXPERIMENT_DIR)

    def _selected_feature_list(frame, *, defect_id, feature_col="feature_name"):
        if frame is None or frame.empty or feature_col not in frame.columns or "defect_id" not in frame.columns:
            return []
        values = frame.loc[frame["defect_id"].astype(str) == str(defect_id), feature_col].dropna().astype(str).tolist()
        return list(dict.fromkeys(values))

    feature_comparison_rows = []
    for defect in DEFECTS:
        defect_id = defect["defect_id"]
        standard_features = _selected_feature_list(standard_one_round_selected_features, defect_id=defect_id)
        block_features = _selected_feature_list(block_selected_features, defect_id=defect_id)
        overlap = sorted(set(standard_features) & set(block_features))
        feature_comparison_rows.append(
            {
                "defect_id": defect_id,
                "standard_selected_features": ", ".join(standard_features),
                "block_selected_features": ", ".join(block_features),
                "overlap_features": ", ".join(overlap),
                "n_standard_selected": len(standard_features),
                "n_block_selected": len(block_features),
                "n_overlap": len(overlap),
            }
        )
    experiment_selected_feature_comparison = pd.DataFrame(feature_comparison_rows)
    write_csv(experiment_selected_feature_comparison, BLOCK_EXPERIMENT_DIR / "experiment_selected_feature_comparison.csv")

    def _last_standard_one_round_curve_row(defect_id):
        if standard_one_round_residual_curve.empty:
            return None
        work = standard_one_round_residual_curve[standard_one_round_residual_curve["defect_id"].astype(str) == str(defect_id)].copy()
        if work.empty:
            return None
        work["round"] = pd.to_numeric(work["round"], errors="coerce")
        work["_curve_order"] = np.arange(len(work))
        return work.sort_values(["round", "_curve_order"]).iloc[-1]

    def _baseline_metric_for(defect_id, *, split, group):
        if baseline_summary is None or baseline_summary.empty:
            return np.nan
        key_defect = "global" if group == "global" else str(defect_id)
        key_group = "global" if group == "global" else str(group)
        rows = baseline_summary[
            (baseline_summary["defect_id"].astype(str) == key_defect)
            & (baseline_summary["split"].astype(str) == str(split))
            & (baseline_summary["group"].astype(str) == key_group)
        ]
        if rows.empty:
            return np.nan
        return pd.to_numeric(rows[EXPERIMENT_METRIC_LOWER], errors="coerce").iloc[0]

    def _curve_row_as_comparison(row, *, defect_id, method, metric_source, selected_features_text):
        valid_bad = row.get(f"valid_bad_{EXPERIMENT_METRIC_LOWER}", np.nan)
        test_bad = row.get(f"test_bad_{EXPERIMENT_METRIC_LOWER}", np.nan)
        valid_global = row.get(f"valid_global_{EXPERIMENT_METRIC_LOWER}", np.nan)
        return {
            "defect_id": defect_id,
            "method": method,
            "metric_source": metric_source,
            "metric_name": EXPERIMENT_METRIC_LOWER,
            "selected_features": selected_features_text,
            "valid_bad_loss_after": valid_bad,
            "valid_bad_loss_after_over_baseline": _safe_ratio(valid_bad, _baseline_metric_for(defect_id, split="valid", group="bad")),
            "test_bad_loss_after": test_bad,
            "test_bad_loss_after_over_baseline": _safe_ratio(test_bad, _baseline_metric_for(defect_id, split="test", group="bad")),
            "valid_global_loss_after": valid_global,
            "valid_global_loss_after_over_baseline": _safe_ratio(valid_global, _baseline_metric_for(defect_id, split="valid", group="global")),
        }

    residual_comparison_rows = []
    block_final_feature_sets_scored = 0
    for defect in DEFECTS:
        defect_id = defect["defect_id"]
        standard_row = _last_standard_one_round_curve_row(defect_id)
        standard_features = _selected_feature_list(standard_one_round_selected_features, defect_id=defect_id)
        if standard_row is not None:
            residual_comparison_rows.append(
                _curve_row_as_comparison(
                    standard_row,
                    defect_id=defect_id,
                    method="standard_feature_by_feature_one_round",
                    metric_source="one round cumulative residual_curve",
                    selected_features_text=", ".join(standard_features),
                )
            )
        else:
            residual_comparison_rows.append(
                {
                    "defect_id": defect_id,
                    "method": "standard_feature_by_feature_one_round",
                    "metric_source": "no selected feature",
                    "selected_features": "",
                }
            )

        block_features = _selected_feature_list(block_selected_features, defect_id=defect_id)
        if block_features:
            block_set_row = _score_residual_feature_set(
                block_features,
                defect_id=defect_id,
                groups=defect_groups[defect_id],
                stage="block_selected_feature_set",
                feature_set_name="block_selected_features",
            )
            block_final_feature_sets_scored += 1
            residual_comparison_rows.append(
                {
                    "defect_id": defect_id,
                    "method": "block_preselection_then_feature",
                    "metric_source": "selected feature set refit",
                    "selected_features": ", ".join(block_features),
                    "metric_name": EXPERIMENT_METRIC_LOWER,
                    "valid_bad_loss_after": block_set_row.get(f"valid_bad_{EXPERIMENT_METRIC_LOWER}_after", np.nan),
                    "valid_bad_loss_after_over_baseline": block_set_row.get(f"valid_bad_{EXPERIMENT_METRIC_LOWER}_after_over_baseline", np.nan),
                    "test_bad_loss_after": block_set_row.get(f"test_bad_{EXPERIMENT_METRIC_LOWER}_after", np.nan),
                    "test_bad_loss_after_over_baseline": block_set_row.get(f"test_bad_{EXPERIMENT_METRIC_LOWER}_after_over_baseline", np.nan),
                    "valid_global_loss_after": block_set_row.get(f"valid_global_{EXPERIMENT_METRIC_LOWER}_after", np.nan),
                    "valid_global_loss_after_over_baseline": block_set_row.get(f"valid_global_{EXPERIMENT_METRIC_LOWER}_after_over_baseline", np.nan),
                }
            )
        else:
            residual_comparison_rows.append(
                {
                    "defect_id": defect_id,
                    "method": "block_preselection_then_feature",
                    "metric_source": "no selected feature",
                    "selected_features": "",
                }
            )
    experiment_residual_comparison = pd.DataFrame(residual_comparison_rows)
    write_csv(experiment_residual_comparison, BLOCK_EXPERIMENT_DIR / "experiment_residual_comparison.csv")

    block_elapsed_seconds = time.perf_counter() - block_experiment_start_time
    experiment_runtime_comparison = pd.DataFrame(
        [
            {
                "method": "standard_feature_by_feature_one_round",
                "elapsed_seconds": standard_one_round_elapsed_seconds,
                "elapsed_minutes": standard_one_round_elapsed_seconds / 60.0,
                "n_candidate_features": len(candidate_cols),
                "n_selected_features": len(standard_one_round_selected_features),
                "stage1_units_scored": len(standard_one_round_rankings),
                "stage2_units_scored": 0,
                "final_feature_sets_scored": 0,
                "total_units_scored": len(standard_one_round_rankings),
            },
            {
                "method": "block_preselection_then_feature",
                "elapsed_seconds": block_elapsed_seconds,
                "elapsed_minutes": block_elapsed_seconds / 60.0,
                "n_candidate_features": len(candidate_cols),
                "n_selected_features": len(block_selected_features),
                "stage1_units_scored": len(block_rankings),
                "stage2_units_scored": len(block_feature_rankings),
                "final_feature_sets_scored": block_final_feature_sets_scored,
                "total_units_scored": len(block_rankings) + len(block_feature_rankings) + block_final_feature_sets_scored,
            },
        ]
    )
    write_csv(experiment_runtime_comparison, BLOCK_EXPERIMENT_DIR / "experiment_runtime_comparison.csv")

    display(experiment_runtime_comparison)
    display(experiment_selected_feature_comparison)
    display(experiment_residual_comparison)
